# Cloud Chaser Kaggle Notebook

This notebook recreates the full Cloud Chaser codebase inside a Kaggle working directory. Segmentation uses the Kaggle Sky-Image Dataset `swimseg-2` cloud masks, while cloud-type classification keeps GCD. Set Kaggle accelerator to GPU before running training cells.

In [ ]:
from pathlib import Path
import os

PROJECT_DIR = Path('/kaggle/working/cloud-chaser')
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

for directory in ['cloud_chaser', 'cloud_chaser/data', 'cloud_chaser/models', 'cloud_chaser/utils', 'configs', 'scripts']:
    Path(directory).mkdir(parents=True, exist_ok=True)

print('Working in', PROJECT_DIR)


## Write Project Files

These cells materialize the Python package and CLI files.

In [ ]:
%%writefile pyproject.toml
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "cloud-chaser"
version = "0.1.0"
description = "Two-stage cloud detection, segmentation, and meteorological classification."
readme = "README.md"
requires-python = ">=3.10,<3.13"
dependencies = [
  "albumentations>=1.4.0",
  "numpy>=1.24",
  "opencv-python>=4.8",
  "pillow>=10.0",
  "pyyaml>=6.0",
  "scikit-learn>=1.3",
  "torch>=2.1",
  "torchvision>=0.16",
  "tqdm>=4.66",
]

[project.optional-dependencies]
dev = ["ruff>=0.5", "pytest>=8.0"]

[project.scripts]
cloud-chaser-train = "train:main"
cloud-chaser-infer = "inference:main"
cloud-chaser-export = "export:main"

[tool.setuptools.packages.find]
include = ["cloud_chaser*"]

[tool.setuptools]
py-modules = ["train", "inference", "export"]

[tool.ruff]
line-length = 100
target-version = "py310"


In [ ]:
%%writefile requirements.txt
albumentations>=1.4.0
matplotlib>=3.8
numpy>=1.24
opencv-python>=4.8
pillow>=10.0
pyyaml>=6.0
scikit-learn>=1.3
torch>=2.1
torchvision>=0.16
tqdm>=4.66


In [ ]:
%%writefile README.md
# Cloud Chaser

Cloud Chaser is now a U-Net-only cloud segmentation and cloud-type classification pipeline. It segments cloud pixels with a PyTorch U-Net trained on SWIMSEG/SkyImage masks, converts connected cloud regions into RGB crops, then classifies each crop into one of the seven GCD cloud categories.

```text
image
  -> U-Net cloud segmentation
  -> threshold cloud probability map
  -> connected cloud components + boxes
  -> RGB bounding-box crop from the original image
  -> GCD cloud-type classifier
  -> mask/box/class overlay and cascade metrics
```

The segmentation mask is used for localization and visualization only. The classifier receives the normal RGB crop from the original image, not a blacked-out mask crop.

## Repository Layout

```text
kaggle/       Kaggle notebook and Kaggle-specific workflow files
local-exec/   Local Python package, CLI scripts, configs, and training code
docs/         Documentation, notes, papers, and architecture plans
venv/         Suggested local virtual environment location
requirements.txt
data/         Local datasets, ignored by git
models/       Local exported model artifacts, ignored by git except docs
README.md
results/      Local training/evaluation/inference outputs, ignored by git except docs
```

## Architecture

### 1. U-Net Segmentation

The segmenter predicts a binary cloud probability map:

```text
input:  RGB image, resized to 224 x 224
output: 1-channel cloud logit map
```

At inference time:

```text
sigmoid(logits)
threshold at unet.threshold
remove connected components smaller than unet.min_area
extract boxes from surviving components
confidence = mean cloud probability inside component
```

### 2. Classification

The classifier is a PyTorch CNN, currently ResNet50 by default, trained on the seven GCD classes:

```text
1_cumulus
2_altocumulus
3_cirrus
4_clearsky
5_stratocumulus
6_cumulonimbus
7_mixed
```

The classifier input is the RGB crop inside the U-Net component bounding box.

### 3. Cascade Evaluation

GCD has image-level class labels but no cloud masks, so final validation uses image-level cascade metrics:

- **segmentation gate accuracy**: cloudy classes should produce at least one U-Net cloud component; clear-sky images should produce none.
- **classifier accuracy given detection**: classification accuracy only among images where a cloud component was produced.
- **cascade accuracy**: end-to-end success after both segmentation gate and classification.

The report outputs are:

```text
results/reports/gcd_val_unet_cascade_bar.png
results/reports/gcd_val_unet_cascade_overlay_samples.jpg
results/reports/gcd_val_unet_cascade_metrics.json
```

On Kaggle, the equivalent paths are under:

```text
/kaggle/working/reports
```

## Local Quick Start

Create a virtual environment:

```bash
python3.12 -m venv venv
source venv/bin/activate
python -m pip install --upgrade pip
python -m pip install -r requirements.txt
python -m pip install -e local-exec
```

Expected local data layout:

```text
data/swimseg-2/**              # SWIMSEG/SkyImage image-mask pairs
data/GCD/train/<class>/*.jpg
data/GCD/test/<class>/*.jpg
```

Run local training from `local-exec/`:

```bash
cd local-exec
python train.py unet --config configs/default.yaml
python train.py classifier --config configs/default.yaml
```

Evaluate:

```bash
python train.py eval-unet --config configs/default.yaml
python train.py eval-classifier --config configs/default.yaml
python scripts/gcd_visual_report.py --config configs/default.yaml --output-dir ../results/reports
```

Run inference:

```bash
python inference.py \
  --config configs/default.yaml \
  --image ../data/example.jpg \
  --output ../results/prediction.jpg
```

## Kaggle Workflow

Use:

```text
kaggle/cloud_chaser_kaggle.ipynb
```

The notebook writes its own working copy to Kaggle, restores checkpoints from the attached `latest-output` dataset when present, resumes training from `last.pt` or `best.pt`, and writes simple outputs to:

```text
/kaggle/working/runs
/kaggle/working/reports
/kaggle/working/checkpoints
/kaggle/working/artifacts
/kaggle/working/prediction.jpg
```

## Checkpoints

Local checkpoints are written under:

```text
results/unet/
results/classifier/
```

The training code resumes in this order:

```text
last.pt -> best.pt -> start fresh
```

For longer experiments, increase the target total epoch counts in:

```text
local-exec/configs/default.yaml
```

The epoch value is a total target, not an additive number. For example, if the U-Net has reached epoch 40 and `epochs: 80`, the next run continues toward epoch 80.


In [ ]:
%%writefile train.py
from __future__ import annotations

import argparse

from cloud_chaser.config import load_config
from cloud_chaser.training import (
    evaluate_classifier,
    evaluate_unet_segmenter,
    train_classifier,
    train_classifier_ssl,
    train_unet_segmenter,
)


def main() -> None:
    parser = argparse.ArgumentParser(description="Cloud Chaser training entrypoint")
    parser.add_argument(
        "stage",
        choices=["unet", "classifier-ssl", "classifier", "eval-classifier", "eval-unet"],
    )
    parser.add_argument("--config", default="configs/default.yaml")
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.stage == "unet":
        train_unet_segmenter(cfg)
    elif args.stage == "classifier-ssl":
        train_classifier_ssl(cfg)
    elif args.stage == "classifier":
        train_classifier(cfg)
    elif args.stage == "eval-classifier":
        evaluate_classifier(cfg)
    elif args.stage == "eval-unet":
        evaluate_unet_segmenter(cfg)


if __name__ == "__main__":
    main()


In [ ]:
%%writefile inference.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2

from cloud_chaser.config import get_device, load_config
from cloud_chaser.inference_pipeline import CloudIdentifier


def main() -> None:
    parser = argparse.ArgumentParser(description="Run cloud segmentation and type classification.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--image", required=True)
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    data_cfg = cfg["data"]
    inf_cfg = cfg["inference"]
    identifier = CloudIdentifier(
        unet_weights=inf_cfg["unet_weights"],
        classifier_weights=inf_cfg["classifier_weights"],
        class_names=data_cfg["classification_classes"],
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=data_cfg["image_size"],
        half=cfg.get("unet", {}).get("half", True),
        crop_padding=inf_cfg["crop_padding"],
    )
    overlay, predictions = identifier.predict(args.image)
    output = Path(args.output) if args.output else Path(inf_cfg["output_dir"]) / Path(args.image).name
    output.parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(str(output), overlay)
    for pred in predictions:
        print(
            f"{pred.class_name}: class={pred.class_confidence:.3f} "
            f"segmenter={pred.segmentation_confidence:.3f} box={pred.box}"
        )
    print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile export.py
from __future__ import annotations

import argparse
import importlib.util
from pathlib import Path

import torch

from cloud_chaser.config import get_device, load_config
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.models.unet import build_unet
from cloud_chaser.utils.checkpoint import load_checkpoint


def _export_torch_model(model: torch.nn.Module, dummy: torch.Tensor, output_path: Path, fmt: str) -> Path:
    output_path.parent.mkdir(parents=True, exist_ok=True)
    if fmt == "torchscript":
        traced = torch.jit.trace(model, dummy)
        traced.save(str(output_path))
    elif fmt == "onnx":
        if importlib.util.find_spec("onnxscript") is None:
            print(f"Skipping ONNX export for {output_path.name}: onnxscript is not installed.")
            return output_path
        torch.onnx.export(
            model,
            dummy,
            str(output_path),
            input_names=["image"],
            output_names=["output"],
            dynamic_axes={"image": {0: "batch"}, "output": {0: "batch"}},
            opset_version=17,
        )
    else:
        raise ValueError("Format must be 'torchscript' or 'onnx'")
    return output_path


def export_unet(cfg: dict, fmt: str, output: str | None = None) -> Path:
    device = get_device(cfg)
    checkpoint_path = cfg["unet"]["checkpoint"]
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    model = build_unet(
        architecture=checkpoint.get("architecture", cfg["unet"].get("architecture", "compact")),
        features=checkpoint.get("features", cfg["unet"].get("features")),
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    image_size = cfg["data"]["image_size"]
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    output_path = Path(output or Path(checkpoint_path).with_suffix(f".{fmt}"))
    return _export_torch_model(model, dummy, output_path, fmt)


def export_classifier(cfg: dict, fmt: str, output: str | None = None) -> Path:
    device = get_device(cfg)
    checkpoint_path = cfg["classifier"]["checkpoint"]
    checkpoint = load_checkpoint(checkpoint_path, map_location=device)
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    image_size = cfg["data"]["image_size"]
    dummy = torch.randn(1, 3, image_size, image_size, device=device)
    output_path = Path(output or Path(checkpoint_path).with_suffix(f".{fmt}"))
    return _export_torch_model(model, dummy, output_path, fmt)


def main() -> None:
    parser = argparse.ArgumentParser(description="Export U-Net segmenter or classifier artifacts.")
    parser.add_argument("target", choices=["unet", "classifier"])
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--format", default="onnx", choices=["onnx", "torchscript"])
    parser.add_argument("--output", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    if args.target == "unet":
        output = export_unet(cfg, args.format, args.output)
        print(f"saved={output}")
    else:
        output = export_classifier(cfg, args.format, args.output)
        print(f"saved={output}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile cloud_chaser/__init__.py
"""Cloud Chaser: cloud segmentation and meteorological type classification."""

__all__ = ["__version__"]
__version__ = "0.1.0"


In [ ]:
%%writefile cloud_chaser/config.py
from __future__ import annotations

from copy import deepcopy
from pathlib import Path
from typing import Any

import yaml


def _deep_update(base: dict[str, Any], override: dict[str, Any]) -> dict[str, Any]:
    for key, value in override.items():
        if isinstance(value, dict) and isinstance(base.get(key), dict):
            base[key] = _deep_update(base[key], value)
        else:
            base[key] = value
    return base


def load_config(path: str | Path = "configs/default.yaml", overrides: dict[str, Any] | None = None) -> dict[str, Any]:
    """Load a YAML config and optionally apply nested dictionary overrides."""
    with Path(path).open("r", encoding="utf-8") as f:
        cfg = yaml.safe_load(f)
    if overrides:
        cfg = _deep_update(deepcopy(cfg), overrides)
    return cfg


def save_yaml(data: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        yaml.safe_dump(data, f, sort_keys=False)


def get_device(cfg: dict[str, Any]) -> str:
    import torch

    requested = cfg.get("project", {}).get("device", "cuda")
    if requested == "cuda" and not torch.cuda.is_available():
        return "cpu"
    return requested


In [ ]:
%%writefile cloud_chaser/data/__init__.py
"""Dataset and preprocessing utilities."""


## Retired ADE20K conversion

This notebook is now U-Net-only. This former generated-file cell is intentionally retired.


In [ ]:
%%writefile cloud_chaser/data/augmentations.py
from __future__ import annotations

import albumentations as A
from albumentations.pytorch import ToTensorV2

IMAGENET_MEAN = (0.485, 0.456, 0.406)
IMAGENET_STD = (0.229, 0.224, 0.225)


def random_resized_crop(image_size: int, scale: tuple[float, float], ratio: tuple[float, float]):
    """Return RandomResizedCrop for both Albumentations 1.x and 2.x APIs."""
    try:
        return A.RandomResizedCrop(size=(image_size, image_size), scale=scale, ratio=ratio)
    except Exception:
        return A.RandomResizedCrop(height=image_size, width=image_size, scale=scale, ratio=ratio)


def classification_train_transforms(
    image_size: int,
    random_shadow_p: float = 0.25,
    gaussian_blur_p: float = 0.2,
    hflip_p: float = 0.5,
    vflip_p: float = 0.15,
) -> A.Compose:
    """Meteorology-aware classification augmentation.

    Flips preserve cloud texture statistics, while blur and shadows simulate focus,
    haze, occlusion, and illumination changes seen in outdoor sky imagery.
    """
    return A.Compose(
        [
            random_resized_crop(image_size, scale=(0.65, 1.0), ratio=(0.85, 1.2)),
            A.HorizontalFlip(p=hflip_p),
            A.VerticalFlip(p=vflip_p),
            A.RandomShadow(p=random_shadow_p),
            A.GaussianBlur(blur_limit=(3, 5), p=gaussian_blur_p),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.12, hue=0.03, p=0.35),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def eval_transforms(image_size: int) -> A.Compose:
    return A.Compose(
        [
            A.Resize(image_size, image_size),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


def contrastive_train_transforms(image_size: int) -> A.Compose:
    return A.Compose(
        [
            random_resized_crop(image_size, scale=(0.45, 1.0), ratio=(0.75, 1.33)),
            A.OneOf(
                [
                    A.Rotate(limit=(90, 90), p=1.0),
                    A.Rotate(limit=(180, 180), p=1.0),
                    A.Rotate(limit=(270, 270), p=1.0),
                ],
                p=0.35,
            ),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.25),
            A.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.04, p=0.6),
            A.GaussNoise(p=0.25),
            A.GaussianBlur(blur_limit=(3, 7), p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ]
    )


In [ ]:
%%writefile cloud_chaser/data/gcd.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
SPLIT_ALIASES = {
    "train": ("train", "training", "Train", "Training"),
    "val": ("val", "valid", "validation", "Val", "Valid", "Validation"),
    "test": ("test", "testing", "Test", "Testing"),
}
CLASS_KEYWORDS = ("cumulus", "altocumulus", "cirrus", "clearsky", "clear", "stratocumulus", "cumulonimbus", "mixed")
FORBIDDEN_CLASSIFIER_ROOTS = ("swimseg", "skyimage", "sky-image", "segmentation", "mask")


@dataclass(frozen=True)
class ImageRecord:
    path: Path
    label: int


def _is_forbidden_classifier_root(path: Path) -> bool:
    text = str(path).lower()
    return any(token in text for token in FORBIDDEN_CLASSIFIER_ROOTS)


def _has_images(path: Path) -> bool:
    return path.exists() and any(p.suffix.lower() in IMAGE_EXTENSIONS for p in path.rglob("*"))


def _image_count(path: Path) -> int:
    return sum(1 for p in path.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)


def _find_split_dir(root: Path, split: str) -> Path | None:
    for name in SPLIT_ALIASES[split]:
        candidate = root / name
        if candidate.exists() and candidate.is_dir():
            return candidate
    return None


def _class_dirs(path: Path) -> list[Path]:
    if not path.exists() or not path.is_dir():
        return []
    return sorted(p for p in path.iterdir() if p.is_dir() and _has_images(p))


def _looks_like_class_root(path: Path) -> bool:
    dirs = _class_dirs(path)
    if len(dirs) < 2:
        return False
    names = " ".join(d.name.lower().replace("_", "") for d in dirs)
    keyword_hits = sum(1 for keyword in CLASS_KEYWORDS if keyword in names)
    return keyword_hits >= 2 or len(dirs) >= 5


def _candidate_roots(root: Path) -> list[Path]:
    candidates: list[Path] = []
    if root.exists():
        candidates.append(root)
        candidates.extend(p for p in root.rglob("*") if p.is_dir())
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend(p for p in kaggle_input.rglob("*") if p.is_dir())
        candidates.extend(p for p in kaggle_input.glob("*") if p.is_dir())
    seen: set[Path] = set()
    unique: list[Path] = []
    for path in candidates:
        if path not in seen:
            unique.append(path)
            seen.add(path)
    return unique


def resolve_gcd_root(root: str | Path) -> Path:
    root = Path(root)
    scored: list[tuple[int, int, Path]] = []
    for path in _candidate_roots(root):
        if _is_forbidden_classifier_root(path):
            continue
        train_dir = _find_split_dir(path, "train")
        if train_dir is not None and _class_dirs(train_dir):
            scored.append((_image_count(train_dir), 0, path))
        elif _looks_like_class_root(path):
            scored.append((_image_count(path), 1, path))
    if scored:
        return sorted(scored, key=lambda item: (-item[0], item[1], len(item[2].parts), str(item[2])))[0][2]

    available = []
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        for path in sorted(kaggle_input.glob("*")):
            available.append(f"{path} images={_image_count(path)}")
    raise FileNotFoundError(
        "Could not find GCD classification images. Attach the GCD dataset or set data.gcd_root to a folder "
        "containing either train/<class>/*.jpg or <class>/*.jpg. Available inputs: " + "; ".join(available)
    )


def discover_classes(root: str | Path) -> list[str]:
    root = resolve_gcd_root(root)
    train_root = _find_split_dir(root, "train") or root
    classes = sorted(p.name for p in _class_dirs(train_root))
    if not classes:
        raise RuntimeError(f"No class folders with images found under {train_root}")
    return classes


def _records_for_split(split_dir: Path | None, classes: list[str]) -> list[ImageRecord]:
    if split_dir is None or not split_dir.exists():
        return []
    records: list[ImageRecord] = []
    class_to_idx = {name: idx for idx, name in enumerate(classes)}
    available = {p.name.lower(): p for p in split_dir.iterdir() if p.is_dir()}
    for class_name in classes:
        class_dir = split_dir / class_name
        if not class_dir.exists():
            class_dir = available.get(class_name.lower())
        if class_dir is None or not class_dir.exists():
            continue
        for path in sorted(class_dir.rglob("*")):
            if path.suffix.lower() in IMAGE_EXTENSIONS:
                records.append(ImageRecord(path=path, label=class_to_idx[class_name]))
    return records


def _split_records(records: list[ImageRecord], split: str, val_fraction: float, seed: int) -> list[ImageRecord]:
    labels = [r.label for r in records]
    indices = list(range(len(records)))
    train_idx, holdout_idx = train_test_split(indices, test_size=val_fraction * 2, random_state=seed, stratify=labels)
    holdout_labels = [records[i].label for i in holdout_idx]
    val_idx, test_idx = train_test_split(holdout_idx, test_size=0.5, random_state=seed, stratify=holdout_labels)
    selected = {"train": train_idx, "val": val_idx, "test": test_idx}[split]
    return [records[i] for i in selected]


def build_gcd_records(
    root: str | Path,
    split: str,
    classes: list[str] | None = None,
    val_fraction: float = 0.15,
    seed: int = 42,
) -> tuple[list[ImageRecord], list[str]]:
    root = resolve_gcd_root(root)
    discovered = discover_classes(root)
    classes = classes or discovered

    train_dir = _find_split_dir(root, "train")
    probe_root = train_dir or root
    probe_records = _records_for_split(probe_root, classes)
    if not probe_records:
        print(f"Configured GCD classes {classes} did not match folders under {probe_root}.")
        print(f"Using discovered classes instead: {discovered}")
        classes = discovered

    if train_dir is None:
        all_records = _records_for_split(root, classes)
        if not all_records:
            raise RuntimeError(f"No GCD images found under class folders in {root}")
        return _split_records(all_records, split, val_fraction, seed), classes

    if split == "test":
        records = _records_for_split(_find_split_dir(root, "test"), classes)
        if records:
            return records, classes
        all_train_records = _records_for_split(train_dir, classes)
        return _split_records(all_train_records, "test", val_fraction, seed), classes

    val_dir = _find_split_dir(root, "val")
    if split == "val" and val_dir is not None:
        records = _records_for_split(val_dir, classes)
        if records:
            return records, classes

    train_records = _records_for_split(train_dir, classes)
    if not train_records:
        raise RuntimeError(f"No GCD train images found under {train_dir}. Check data.gcd_root.")
    labels = [r.label for r in train_records]
    indices = list(range(len(train_records)))
    train_idx, val_idx = train_test_split(indices, test_size=val_fraction, random_state=seed, stratify=labels)
    selected = train_idx if split == "train" else val_idx
    return [train_records[i] for i in selected], classes


class GCDDataset(Dataset):
    def __init__(
        self,
        root: str | Path,
        split: str,
        transform=None,
        classes: list[str] | None = None,
        val_fraction: float = 0.15,
        seed: int = 42,
    ) -> None:
        self.records, self.classes = build_gcd_records(root, split, classes, val_fraction, seed)
        self.transform = transform
        print(f"GCD {split}: {len(self.records)} images, classes={self.classes}")

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int):
        record = self.records[index]
        image = cv2.cvtColor(cv2.imread(str(record.path)), cv2.COLOR_BGR2RGB)
        if self.transform:
            image = self.transform(image=image)["image"]
        else:
            image = Image.open(record.path).convert("RGB")
        return image, record.label


In [ ]:
%%writefile cloud_chaser/data/swimseg.py
from __future__ import annotations

import csv
import re
from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import torch
from torch.utils.data import Dataset
from tqdm import tqdm

from cloud_chaser.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from cloud_chaser.data.gcd import IMAGE_EXTENSIONS

MASK_TOKENS = (
    "mask",
    "masks",
    "gt",
    "gtmaps",
    "groundtruth",
    "ground_truth",
    "truth",
    "label",
    "labels",
    "annotation",
    "annotations",
    "segmentation",
    "binary",
)


@dataclass(frozen=True)
class SwimsegPair:
    image_path: Path
    mask_path: Path


def _is_probable_binary_mask(path: Path) -> bool:
    image = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        return False
    sample = image[:: max(1, image.shape[0] // 256), :: max(1, image.shape[1] // 256)]
    unique = np.unique(sample)
    if len(unique) <= 8:
        return True
    low_high = ((sample < 16) | (sample > 239)).mean()
    return bool(low_high > 0.97)


def _looks_like_mask_path(path: Path) -> bool:
    joined = "/".join(part.lower() for part in path.parts)
    return any(token in joined for token in MASK_TOKENS) or _is_probable_binary_mask(path)


def _normal_stem(path: Path) -> str:
    stem = path.stem.lower()
    stem = re.sub(
        r"(_|-)?(mask|gt|gtmap|groundtruth|ground_truth|truth|label|annotation|segmentation|binary)$",
        "",
        stem,
    )
    return re.sub(r"[^a-z0-9]+", "", stem)


def discover_swimseg_pairs(root: str | Path) -> list[SwimsegPair]:
    root = Path(root)
    if not root.exists():
        raise FileNotFoundError(f"SWIMSEG root does not exist: {root}")

    files = sorted(p for p in root.rglob("*") if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS)
    if not files:
        raise RuntimeError(f"No image files found under {root}")

    mask_set = {p for p in files if _looks_like_mask_path(p)}
    mask_files = sorted(mask_set)
    image_files = sorted(p for p in files if p not in mask_set)

    if not image_files or not mask_files:
        mask_set = {p for p in files if _is_probable_binary_mask(p)}
        mask_files = sorted(mask_set)
        image_files = sorted(p for p in files if p not in mask_set)

    masks_by_key: dict[str, list[Path]] = {}
    for mask in mask_files:
        masks_by_key.setdefault(_normal_stem(mask), []).append(mask)

    pairs: list[SwimsegPair] = []
    used_masks: set[Path] = set()
    for image in image_files:
        candidates = [m for m in masks_by_key.get(_normal_stem(image), []) if m not in used_masks]
        if candidates:
            mask = candidates[0]
            pairs.append(SwimsegPair(image, mask))
            used_masks.add(mask)

    # Some packaged SWIMSEG variants keep images and masks aligned by sorted order.
    if len(pairs) < min(len(image_files), len(mask_files)) * 0.5:
        pairs = []
        for image, mask in zip(sorted(image_files), sorted(mask_files), strict=False):
            img = cv2.imread(str(image), cv2.IMREAD_COLOR)
            msk = cv2.imread(str(mask), cv2.IMREAD_GRAYSCALE)
            if img is not None and msk is not None and img.shape[:2] == msk.shape[:2]:
                pairs.append(SwimsegPair(image, mask))

    if not pairs:
        raise RuntimeError(
            f"Could not pair SWIMSEG images and masks under {root}. "
            "Inspect the dataset tree and update data.swimseg_root."
        )
    print(f"Discovered {len(pairs)} SWIMSEG image/mask pairs under {root}")
    return pairs


def _binary_cloud_mask(mask_path: Path, invert: bool = False) -> np.ndarray:
    mask = np.array(Image.open(mask_path).convert("L"))
    binary = mask > 127
    if invert:
        binary = ~binary
    return binary.astype(np.uint8)


class SwimsegMaskDataset(Dataset):
    def __init__(
        self,
        prepared_dir: str | Path,
        split: str,
        image_size: int,
    ) -> None:
        self.prepared_dir = Path(prepared_dir)
        self.split = split
        self.image_size = image_size
        manifest = self.prepared_dir / "manifest.csv"
        if not manifest.exists():
            raise FileNotFoundError(f"SWIMSEG manifest not found: {manifest}")
        self.records: list[tuple[Path, Path]] = []
        with manifest.open("r", newline="", encoding="utf-8") as f:
            for row in csv.DictReader(f):
                if row["split"] == split:
                    self.records.append((Path(row["image"]), Path(row["mask"])))
        if not self.records:
            raise RuntimeError(f"No SWIMSEG records found for split={split} in {manifest}")
        self.mean = np.array(IMAGENET_MEAN, dtype=np.float32)
        self.std = np.array(IMAGENET_STD, dtype=np.float32)

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image_path, mask_path = self.records[index]
        image = cv2.cvtColor(cv2.imread(str(image_path), cv2.IMREAD_COLOR), cv2.COLOR_BGR2RGB)
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if image is None or mask is None:
            raise FileNotFoundError(f"Could not read SWIMSEG pair: {image_path}, {mask_path}")
        image = cv2.resize(image, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        mask = cv2.resize(mask, (self.image_size, self.image_size), interpolation=cv2.INTER_NEAREST)
        image = image.astype(np.float32) / 255.0
        image = (image - self.mean) / self.std
        mask = (mask > 127).astype(np.float32)
        image_tensor = torch.from_numpy(image.transpose(2, 0, 1)).float()
        mask_tensor = torch.from_numpy(mask[None]).float()
        return image_tensor, mask_tensor


def _safe_link_or_copy(source: Path, target: Path) -> None:
    target.parent.mkdir(parents=True, exist_ok=True)
    if target.exists():
        return
    try:
        target.symlink_to(source.resolve())
    except OSError:
        image = cv2.imread(str(source), cv2.IMREAD_UNCHANGED)
        if image is None:
            raise FileNotFoundError(source)
        cv2.imwrite(str(target), image)


def prepare_swimseg_masks(
    root: str | Path,
    output_dir: str | Path,
    val_fraction: float = 0.1,
    test_fraction: float = 0.1,
    seed: int = 42,
    invert_masks: bool = False,
) -> Path:
    output_dir = Path(output_dir)
    pairs = discover_swimseg_pairs(root)
    indices = list(range(len(pairs)))
    train_idx, holdout_idx = train_test_split(
        indices,
        test_size=val_fraction + test_fraction,
        random_state=seed,
    )
    relative_test = test_fraction / max(val_fraction + test_fraction, 1e-9)
    val_idx, test_idx = train_test_split(holdout_idx, test_size=relative_test, random_state=seed)
    split_indices = {"train": train_idx, "val": val_idx, "test": test_idx}

    manifest_path = output_dir / "manifest.csv"
    manifest_path.parent.mkdir(parents=True, exist_ok=True)
    with manifest_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=["split", "image", "mask"])
        writer.writeheader()
        for split, split_idx in split_indices.items():
            for idx in tqdm(split_idx, desc=f"Preparing SWIMSEG {split}"):
                pair = pairs[idx]
                image = cv2.imread(str(pair.image_path), cv2.IMREAD_COLOR)
                if image is None:
                    continue
                h, w = image.shape[:2]
                binary = _binary_cloud_mask(pair.mask_path, invert=invert_masks)
                if binary.shape != (h, w):
                    binary = cv2.resize(binary, (w, h), interpolation=cv2.INTER_NEAREST)

                target_image = output_dir / "images" / split / pair.image_path.name
                target_mask = output_dir / "masks" / split / f"{pair.image_path.stem}.png"
                _safe_link_or_copy(pair.image_path, target_image)
                target_mask.parent.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(target_mask), binary * 255)
                writer.writerow({"split": split, "image": str(target_image), "mask": str(target_mask)})

    return manifest_path


## Retired external evaluator

U-Net evaluation is handled by `train.py eval-unet`.


In [ ]:
%%writefile cloud_chaser/inference_pipeline.py
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import cv2
import numpy as np
import torch

from cloud_chaser.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from cloud_chaser.data.augmentations import eval_transforms
from cloud_chaser.models.classifier import CloudClassifier
from cloud_chaser.models.unet import build_unet
from cloud_chaser.utils.checkpoint import load_checkpoint
from cloud_chaser.utils.visualization import overlay_instance


def display_class_name(class_name: str) -> str:
    name = class_name.split("_", 1)[1] if "_" in class_name and class_name[0].isdigit() else class_name
    return name.replace("_", " ").title()


@dataclass
class CloudPrediction:
    box: tuple[int, int, int, int]
    segmentation_confidence: float
    class_name: str
    class_confidence: float


class CloudIdentifier:
    def __init__(
        self,
        unet_weights: str | Path,
        classifier_weights: str | Path,
        class_names: list[str] | None = None,
        unet_threshold: float = 0.45,
        unet_min_area: int = 256,
        device: str = "cuda",
        image_size: int = 224,
        half: bool = True,
        crop_padding: int = 12,
    ) -> None:
        self.device = device if device != "cuda" or torch.cuda.is_available() else "cpu"
        self.half = half and self.device != "cpu"
        self.crop_padding = crop_padding
        self.image_size = image_size
        self.unet_threshold = unet_threshold
        self.unet_min_area = unet_min_area
        checkpoint = load_checkpoint(unet_weights, map_location=self.device)
        self.unet = build_unet(
            architecture=checkpoint.get("architecture", "compact"),
            features=checkpoint.get("features"),
        ).to(self.device)
        self.unet.load_state_dict(checkpoint["model"])
        self.unet.eval()
        self.transform = eval_transforms(image_size)

        classifier_path = Path(classifier_weights)
        if classifier_path.suffix in {".torchscript", ".ts"}:
            if class_names is None:
                raise ValueError("class_names are required when loading a TorchScript classifier.")
            self.classes = class_names
            self.classifier = torch.jit.load(str(classifier_path), map_location=self.device).to(self.device)
        else:
            checkpoint = load_checkpoint(classifier_path, map_location=self.device)
            self.classes = checkpoint["classes"]
            self.classifier = CloudClassifier(
                num_classes=len(self.classes),
                backbone=checkpoint["backbone"],
                dropout=0.0,
                pretrained=False,
            ).to(self.device)
            self.classifier.load_state_dict(checkpoint["model"])
        self.classifier.eval()

    def _crop_instances(
        self,
        image_rgb: np.ndarray,
        masks: torch.Tensor,
        boxes: torch.Tensor,
    ) -> list[torch.Tensor]:
        h, w = image_rgb.shape[:2]
        crops: list[torch.Tensor] = []
        for _, box_tensor in zip(masks, boxes, strict=False):
            x1, y1, x2, y2 = [int(v) for v in box_tensor.tolist()]
            x1 = max(0, x1 - self.crop_padding)
            y1 = max(0, y1 - self.crop_padding)
            x2 = min(w, x2 + self.crop_padding)
            y2 = min(h, y2 + self.crop_padding)
            crop = image_rgb[y1:y2, x1:x2]
            crops.append(self.transform(image=crop)["image"])
        return crops

    def _unet_instances(self, image_rgb: np.ndarray) -> tuple[torch.Tensor, torch.Tensor, list[float]]:
        h, w = image_rgb.shape[:2]
        resized = cv2.resize(image_rgb, (self.image_size, self.image_size), interpolation=cv2.INTER_AREA)
        x = resized.astype(np.float32) / 255.0
        x = (x - np.array(IMAGENET_MEAN, dtype=np.float32)) / np.array(IMAGENET_STD, dtype=np.float32)
        tensor = torch.from_numpy(x.transpose(2, 0, 1))[None].float().to(self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.half):
            prob = torch.sigmoid(self.unet(tensor))[0, 0].detach().float().cpu().numpy()
        prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
        binary = prob >= self.unet_threshold
        num_labels, labels = cv2.connectedComponents(binary.astype(np.uint8), connectivity=8)
        masks: list[np.ndarray] = []
        boxes: list[list[float]] = []
        scores: list[float] = []
        for component_id in range(1, num_labels):
            mask = labels == component_id
            area = int(mask.sum())
            if area < self.unet_min_area:
                continue
            ys, xs = np.where(mask)
            x1, x2 = int(xs.min()), int(xs.max()) + 1
            y1, y2 = int(ys.min()), int(ys.max()) + 1
            masks.append(mask)
            boxes.append([x1, y1, x2, y2])
            scores.append(float(prob[mask].mean()))
        if not masks:
            return torch.empty((0, h, w), dtype=torch.bool), torch.empty((0, 4)), []
        return torch.from_numpy(np.stack(masks)).bool(), torch.tensor(boxes, dtype=torch.float32), scores

    @torch.no_grad()
    def predict(self, image_path: str | Path) -> tuple[np.ndarray, list[CloudPrediction]]:
        image_bgr = cv2.imread(str(image_path))
        if image_bgr is None:
            raise FileNotFoundError(f"Could not read image: {image_path}")
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        h, w = image_rgb.shape[:2]

        masks, boxes, segmentation_scores = self._unet_instances(image_rgb)

        if len(boxes) == 0:
            return image_bgr, []

        crops = self._crop_instances(image_rgb, masks, boxes)
        batch = torch.stack(crops).to(self.device)
        with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=self.half):
            probs = torch.softmax(self.classifier(batch), dim=1)
        confs, labels = probs.max(dim=1)

        overlay = image_bgr.copy()
        predictions: list[CloudPrediction] = []
        for i, (box_tensor, seg_score, label_idx, cls_score) in enumerate(
            zip(boxes, segmentation_scores, labels, confs, strict=False)
        ):
            box = tuple(int(v) for v in box_tensor.tolist())
            class_name = display_class_name(self.classes[int(label_idx)])
            class_conf = float(cls_score.detach().cpu())
            mask = masks[i].detach().cpu().numpy().astype(bool)
            overlay = overlay_instance(
                overlay,
                mask,
                box,
                class_name,
                class_conf,
                color_index=int(label_idx),
            )
            predictions.append(
                CloudPrediction(
                    box=box,
                    segmentation_confidence=float(seg_score),
                    class_name=class_name,
                    class_confidence=class_conf,
                )
            )
        return overlay, predictions


In [ ]:
%%writefile cloud_chaser/models/__init__.py
"""Model definitions and wrappers."""


In [ ]:
%%writefile cloud_chaser/models/classifier.py
from __future__ import annotations

from dataclasses import dataclass
from typing import Literal

import torch
from torch import nn
from torchvision import models

BackboneName = Literal["resnet50", "efficientnet_b0", "densenet121"]


@dataclass(frozen=True)
class BackboneBundle:
    encoder: nn.Module
    features_dim: int


class DenseNetEncoder(nn.Module):
    def __init__(self, model: models.DenseNet) -> None:
        super().__init__()
        self.features = model.features
        self.pool = nn.AdaptiveAvgPool2d((1, 1))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = torch.relu(self.features(x))
        x = self.pool(x)
        return torch.flatten(x, 1)


def build_encoder(backbone: BackboneName = "resnet50", pretrained: bool = True) -> BackboneBundle:
    if backbone == "resnet50":
        weights = models.ResNet50_Weights.IMAGENET1K_V2 if pretrained else None
        model = models.resnet50(weights=weights)
        features_dim = model.fc.in_features
        model.fc = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        features_dim = model.classifier[1].in_features
        model.classifier = nn.Identity()
        return BackboneBundle(model, features_dim)
    if backbone == "densenet121":
        weights = models.DenseNet121_Weights.IMAGENET1K_V1 if pretrained else None
        model = models.densenet121(weights=weights)
        return BackboneBundle(DenseNetEncoder(model), model.classifier.in_features)
    raise ValueError(f"Unsupported backbone: {backbone}")


class CloudClassifier(nn.Module):
    def __init__(
        self,
        num_classes: int,
        backbone: BackboneName = "resnet50",
        dropout: float = 0.2,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        bundle = build_encoder(backbone, pretrained=pretrained)
        self.backbone_name = backbone
        self.encoder = bundle.encoder
        self.head = nn.Sequential(
            nn.Dropout(p=dropout),
            nn.Linear(bundle.features_dim, num_classes),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.head(self.encoder(x))

    def freeze_encoder(self, freeze: bool = True) -> None:
        for param in self.encoder.parameters():
            param.requires_grad = not freeze


class ProjectionHead(nn.Module):
    def __init__(self, in_dim: int, hidden_dim: int = 2048, out_dim: int = 128) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden_dim),
            nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class ContrastiveCloudEncoder(nn.Module):
    """ResNet-style encoder plus MLP projection head for CSSL/MoCo pretraining."""

    def __init__(
        self,
        backbone: BackboneName = "resnet50",
        projection_dim: int = 128,
        pretrained: bool = True,
    ) -> None:
        super().__init__()
        bundle = build_encoder(backbone, pretrained=pretrained)
        self.backbone_name = backbone
        self.encoder = bundle.encoder
        self.projector = ProjectionHead(bundle.features_dim, out_dim=projection_dim)
        self.features_dim = bundle.features_dim

    def forward_features(self, x: torch.Tensor) -> torch.Tensor:
        return self.encoder(x)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.projector(self.forward_features(x))


## Retired external model wrapper

The segmentation model is now implemented entirely in `cloud_chaser/models/unet.py`.


In [ ]:
%%writefile cloud_chaser/models/unet.py
from __future__ import annotations

from typing import Literal

import torch
from torch import nn
import torch.nn.functional as F

UNetArchitecture = Literal[
    "compact",
    "standard",
    "dilated",
    "dilated_aspp",
    "bicubic",
    "improved",
]


class DoubleConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DilatedConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, dilation: int = 2) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=3,
                padding=dilation,
                dilation=dilation,
                bias=False,
            ),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class ASPP(nn.Module):
    def __init__(self, channels: int, rates: tuple[int, ...] = (1, 3, 5)) -> None:
        super().__init__()
        branches = []
        for rate in rates:
            branches.append(
                nn.Sequential(
                    nn.Conv2d(
                        channels,
                        channels,
                        kernel_size=3,
                        padding=rate,
                        dilation=rate,
                        bias=False,
                    ),
                    nn.BatchNorm2d(channels),
                    nn.ReLU(inplace=True),
                )
            )
        branches.append(
            nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(channels),
                nn.ReLU(inplace=True),
            )
        )
        self.branches = nn.ModuleList(branches)
        self.project = nn.Sequential(
            nn.Conv2d(channels * len(branches), channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.project(torch.cat([branch(x) for branch in self.branches], dim=1))


class DilateBlock(nn.Module):
    def __init__(self, in_channels: int, out_channels: int, use_aspp: bool) -> None:
        super().__init__()
        self.conv1 = DilatedConv(in_channels, out_channels, dilation=2)
        self.aspp = ASPP(out_channels) if use_aspp else nn.Identity()
        self.conv2 = DilatedConv(out_channels, out_channels, dilation=2)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.conv2(self.aspp(self.conv1(x)))


class DepthwiseSeparableConv(nn.Module):
    def __init__(self, in_channels: int, out_channels: int) -> None:
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, in_channels, kernel_size=3, padding=1, groups=in_channels, bias=False),
            nn.BatchNorm2d(in_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class DSResidualPath(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.conv1 = DepthwiseSeparableConv(channels, channels)
        self.conv2 = DepthwiseSeparableConv(channels, channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x + self.conv2(self.conv1(x))


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8) -> None:
        super().__init__()
        hidden = max(channels // reduction, 4)
        self.mlp = nn.Sequential(
            nn.Conv2d(channels, hidden, kernel_size=1, bias=False),
            nn.ReLU(inplace=True),
            nn.Conv2d(hidden, channels, kernel_size=1, bias=False),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        avg = F.adaptive_avg_pool2d(x, 1)
        mx = F.adaptive_max_pool2d(x, 1)
        return x * torch.sigmoid(self.mlp(avg) + self.mlp(mx))


class SpatialAttention(nn.Module):
    def __init__(self, channels: int) -> None:
        super().__init__()
        self.attn = nn.Sequential(
            DepthwiseSeparableConv(channels, channels),
            nn.Conv2d(channels, 1, kernel_size=1),
            nn.Sigmoid(),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return x * self.attn(x)


class ImprovedCSAM(nn.Module):
    """Practical PyTorch Im-CSAM approximation: channel then spatial attention."""

    def __init__(self, channels: int) -> None:
        super().__init__()
        self.channel = ChannelAttention(channels)
        self.spatial = SpatialAttention(channels)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.spatial(self.channel(x))


class SkipProcessor(nn.Module):
    def __init__(self, channels: int, enabled: bool) -> None:
        super().__init__()
        self.block = nn.Sequential(DSResidualPath(channels), ImprovedCSAM(channels)) if enabled else nn.Identity()

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.block(x)


class UpBlock(nn.Module):
    def __init__(self, in_channels: int, skip_channels: int, out_channels: int, mode: str) -> None:
        super().__init__()
        self.mode = mode
        if mode == "transpose":
            self.up = nn.ConvTranspose2d(in_channels, out_channels, kernel_size=2, stride=2)
        elif mode == "bicubic":
            self.up = nn.Sequential(
                nn.Upsample(scale_factor=2, mode="bicubic", align_corners=False),
                nn.Conv2d(in_channels, out_channels, kernel_size=1, bias=False),
                nn.BatchNorm2d(out_channels),
                nn.ReLU(inplace=True),
            )
        else:
            raise ValueError(f"Unsupported upsample mode: {mode}")
        self.conv = DoubleConv(out_channels + skip_channels, out_channels)

    def forward(self, x: torch.Tensor, skip: torch.Tensor) -> torch.Tensor:
        x = self.up(x)
        if x.shape[-2:] != skip.shape[-2:]:
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear", align_corners=False)
        return self.conv(torch.cat([skip, x], dim=1))


class CloudUNet(nn.Module):
    """Configurable U-Net family for binary cloud semantic segmentation."""

    def __init__(
        self,
        architecture: UNetArchitecture = "compact",
        features: list[int] | tuple[int, ...] | None = None,
    ) -> None:
        super().__init__()
        if features is None:
            features = (64, 128, 256, 512) if architecture == "standard" else (32, 64, 128, 256)
        if len(features) != 4:
            raise ValueError("CloudUNet expects four encoder feature levels.")
        self.architecture = architecture
        self.features = list(features)
        use_dilated = architecture in {"dilated", "dilated_aspp", "bicubic", "improved"}
        use_aspp = architecture in {"dilated_aspp", "bicubic", "improved"}
        use_bicubic = architecture in {"bicubic", "improved"}
        use_attention_skip = architecture == "improved"

        encoder_blocks: list[nn.Module] = []
        in_channels = 3
        for out_channels in self.features:
            if use_dilated:
                encoder_blocks.append(DilateBlock(in_channels, out_channels, use_aspp=use_aspp))
            else:
                encoder_blocks.append(DoubleConv(in_channels, out_channels))
            in_channels = out_channels
        self.encoder = nn.ModuleList(encoder_blocks)
        self.pool = nn.MaxPool2d(2)
        self.bottleneck = DoubleConv(self.features[-1], self.features[-1] * 2)
        self.skip_processors = nn.ModuleList(
            [SkipProcessor(channels, enabled=use_attention_skip) for channels in self.features]
        )

        up_mode = "bicubic" if use_bicubic else "transpose"
        decoder_blocks: list[nn.Module] = []
        current_channels = self.features[-1] * 2
        for skip_channels in reversed(self.features):
            decoder_blocks.append(UpBlock(current_channels, skip_channels, skip_channels, up_mode))
            current_channels = skip_channels
        self.decoder = nn.ModuleList(decoder_blocks)
        self.out = nn.Conv2d(self.features[0], 1, kernel_size=1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        skips: list[torch.Tensor] = []
        for block in self.encoder:
            x = block(x)
            skips.append(x)
            x = self.pool(x)
        x = self.bottleneck(x)
        processed = [processor(skip) for processor, skip in zip(self.skip_processors, skips, strict=True)]
        for block, skip in zip(self.decoder, reversed(processed), strict=True):
            x = block(x, skip)
        return self.out(x)


def build_unet(
    architecture: UNetArchitecture = "compact",
    features: list[int] | tuple[int, ...] | None = None,
) -> CloudUNet:
    return CloudUNet(architecture=architecture, features=features)


UNET_EXPERIMENTS: dict[str, dict[str, object]] = {
    "baseline_a_compact": {"architecture": "compact", "features": [32, 64, 128, 256]},
    "baseline_b_medium": {"architecture": "standard", "features": [64, 128, 256, 512]},
    "model_c_dilated": {"architecture": "dilated", "features": [64, 128, 256, 512]},
    "model_d_dilated_aspp": {"architecture": "dilated_aspp", "features": [64, 128, 256, 512]},
    "model_e_bicubic": {"architecture": "bicubic", "features": [64, 128, 256, 512]},
    "model_f_improved": {"architecture": "improved", "features": [64, 128, 256, 512]},
}


In [ ]:
%%writefile cloud_chaser/training.py
from __future__ import annotations

from pathlib import Path

import cv2
import torch
import torch.nn.functional as F
from torch import nn
from torch.utils.data import DataLoader
from tqdm import tqdm

from cloud_chaser.config import get_device
from cloud_chaser.data.augmentations import (
    classification_train_transforms,
    contrastive_train_transforms,
    eval_transforms,
)
from cloud_chaser.data.gcd import GCDDataset, build_gcd_records
from cloud_chaser.data.swimseg import SwimsegMaskDataset, prepare_swimseg_masks
from cloud_chaser.models.classifier import CloudClassifier, ContrastiveCloudEncoder
from cloud_chaser.models.unet import CloudUNet, build_unet
from cloud_chaser.utils.checkpoint import load_checkpoint, save_checkpoint
from cloud_chaser.utils.metrics import classification_metrics
from cloud_chaser.utils.seed import seed_everything


def _segmentation_scores(logits: torch.Tensor, masks: torch.Tensor) -> tuple[float, float]:
    preds = torch.sigmoid(logits) > 0.5
    targets = masks > 0.5
    intersection = (preds & targets).sum(dim=(1, 2, 3)).float()
    union = (preds | targets).sum(dim=(1, 2, 3)).float().clamp_min(1.0)
    pred_sum = preds.sum(dim=(1, 2, 3)).float()
    target_sum = targets.sum(dim=(1, 2, 3)).float()
    iou = (intersection / union).mean().item()
    dice = ((2 * intersection + 1.0) / (pred_sum + target_sum + 1.0)).mean().item()
    return iou, dice


def _unet_kwargs(unet_cfg: dict) -> dict[str, object]:
    return {
        "architecture": unet_cfg.get("architecture", "compact"),
        "features": unet_cfg.get("features"),
    }


def _unet_output_dir(cfg: dict, experiment_name: str | None = None) -> Path:
    root = Path(cfg["project"]["output_dir"]) / "unet"
    return root / experiment_name if experiment_name else root


def _run_unet_epoch(
    model: nn.Module,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: torch.optim.Optimizer | None = None,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    scaler = torch.cuda.amp.GradScaler(enabled=training and device == "cuda")
    total_loss = 0.0
    total_iou = 0.0
    total_dice = 0.0
    for images, masks in tqdm(loader, desc="unet-train" if training else "unet-eval"):
        images = images.to(device, non_blocking=True)
        masks = masks.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                logits = model(images)
                bce = criterion(logits, masks)
                probs = torch.sigmoid(logits)
                intersection = (probs * masks).sum(dim=(1, 2, 3))
                dice_loss = 1 - ((2 * intersection + 1) / (probs.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3)) + 1)).mean()
                loss = bce + dice_loss
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        iou, dice = _segmentation_scores(logits.detach(), masks.detach())
        total_loss += float(loss.detach().cpu())
        total_iou += iou
        total_dice += dice
    count = max(1, len(loader))
    return {"loss": total_loss / count, "miou": total_iou / count, "dice": total_dice / count}


def train_unet_segmenter(cfg: dict, experiment_name: str | None = None, unet_override: dict | None = None) -> Path:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    unet_cfg = {**cfg["unet"], **(unet_override or {})}
    prepare_swimseg_masks(
        root=data_cfg["swimseg_root"],
        output_dir=data_cfg["prepared_seg_dir"],
        val_fraction=data_cfg.get("seg_val_fraction", 0.1),
        test_fraction=data_cfg.get("seg_test_fraction", 0.1),
        seed=cfg["project"]["seed"],
        invert_masks=data_cfg.get("swimseg_invert_masks", False),
    )
    train_ds = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "train", data_cfg["image_size"])
    val_ds = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "val", data_cfg["image_size"])
    train_loader = DataLoader(
        train_ds,
        batch_size=unet_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=unet_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = build_unet(**_unet_kwargs(unet_cfg)).to(device)
    criterion = nn.BCEWithLogitsLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=unet_cfg["lr"], weight_decay=unet_cfg["weight_decay"])
    output_dir = _unet_output_dir(cfg, experiment_name)
    last_checkpoint = output_dir / "last.pt"
    best_checkpoint = output_dir / "best.pt"
    best_miou = -1.0
    start_epoch = 0
    resume_checkpoint = last_checkpoint if last_checkpoint.exists() else best_checkpoint if best_checkpoint.exists() else None
    if resume_checkpoint is not None:
        checkpoint = load_checkpoint(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model"])
        if "optimizer" in checkpoint:
            optimizer.load_state_dict(checkpoint["optimizer"])
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        best_miou = float(checkpoint.get("best_miou", checkpoint.get("val_metrics", {}).get("miou", -1.0)))
        print(f"Resuming U-Net from {resume_checkpoint} at epoch {start_epoch + 1}")
    for epoch in range(start_epoch, unet_cfg["epochs"]):
        train_metrics = _run_unet_epoch(model, train_loader, criterion, device, optimizer)
        val_metrics = _run_unet_epoch(model, val_loader, criterion, device)
        print(
            f"unet_epoch={epoch + 1} train_loss={train_metrics['loss']:.4f} "
            f"val_miou={val_metrics['miou']:.4f} val_dice={val_metrics['dice']:.4f}"
        )
        payload = {
            "epoch": epoch,
            "architecture": unet_cfg.get("architecture", "compact"),
            "features": unet_cfg.get("features"),
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "best_miou": max(best_miou, val_metrics["miou"]),
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if val_metrics["miou"] > best_miou:
            best_miou = val_metrics["miou"]
            save_checkpoint(payload, output_dir / "best.pt")
    return output_dir / "best.pt"


def evaluate_unet_segmenter(
    cfg: dict,
    checkpoint_path: str | Path | None = None,
    unet_override: dict | None = None,
) -> dict[str, float]:
    device = get_device(cfg)
    data_cfg = cfg["data"]
    checkpoint = load_checkpoint(checkpoint_path or cfg["unet"]["checkpoint"], map_location=device)
    dataset = SwimsegMaskDataset(data_cfg["prepared_seg_dir"], "test", data_cfg["image_size"])
    loader = DataLoader(
        dataset,
        batch_size=cfg["unet"]["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    unet_cfg = {**cfg["unet"], **(unet_override or {})}
    architecture = checkpoint.get("architecture", unet_cfg.get("architecture", "compact"))
    features = checkpoint.get("features", unet_cfg.get("features"))
    model = build_unet(architecture=architecture, features=features).to(device)
    model.load_state_dict(checkpoint["model"])
    metrics = _run_unet_epoch(model, loader, nn.BCEWithLogitsLoss(), device)
    print(metrics)
    return metrics


def _run_classifier_epoch(
    model: CloudClassifier,
    loader: DataLoader,
    criterion: nn.Module,
    device: str,
    optimizer: torch.optim.Optimizer | None = None,
    amp: bool = True,
) -> dict[str, float]:
    training = optimizer is not None
    model.train(training)
    total_loss = 0.0
    all_logits: list[torch.Tensor] = []
    all_targets: list[torch.Tensor] = []
    scaler = torch.cuda.amp.GradScaler(enabled=training and amp and device == "cuda")
    for images, targets in tqdm(loader, desc="train" if training else "eval"):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with torch.set_grad_enabled(training):
            if training:
                optimizer.zero_grad(set_to_none=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=amp and device == "cuda"):
                logits = model(images)
                loss = criterion(logits, targets)
            if training:
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
        total_loss += float(loss.detach().cpu())
        all_logits.append(logits.detach().cpu())
        all_targets.append(targets.detach().cpu())
    logits = torch.cat(all_logits)
    targets = torch.cat(all_targets)
    metrics = classification_metrics(logits, targets)
    metrics["loss"] = total_loss / max(1, len(loader))
    return metrics


class ContrastiveGCDDataset(torch.utils.data.Dataset):
    def __init__(self, root: str | Path, image_size: int, classes: list[str], val_fraction: float, seed: int) -> None:
        records, _ = build_gcd_records(root, "train", classes=classes, val_fraction=val_fraction, seed=seed)
        self.paths = [record.path for record in records]
        self.transform = contrastive_train_transforms(image_size)

    def __len__(self) -> int:
        return len(self.paths)

    def __getitem__(self, index: int) -> tuple[torch.Tensor, torch.Tensor]:
        image_bgr = cv2.imread(str(self.paths[index]), cv2.IMREAD_COLOR)
        if image_bgr is None:
            raise FileNotFoundError(self.paths[index])
        image = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        q = self.transform(image=image)["image"]
        k = self.transform(image=image)["image"]
        return q, k


def _dequeue_and_enqueue(queue: torch.Tensor, keys: torch.Tensor, ptr: int) -> int:
    batch_size = keys.shape[0]
    queue_size = queue.shape[0]
    if batch_size >= queue_size:
        queue.copy_(keys[-queue_size:])
        return 0
    end = ptr + batch_size
    if end <= queue_size:
        queue[ptr:end] = keys
    else:
        first = queue_size - ptr
        queue[ptr:] = keys[:first]
        queue[: end - queue_size] = keys[first:]
    return end % queue_size


def train_classifier_ssl(cfg: dict) -> Path:
    """MoCo-style CSSL pretraining from the attached ground-cloud classification paper."""
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    ssl_cfg = cls_cfg.get("ssl", {})
    output_dir = Path(cfg["project"]["output_dir"]) / "classifier_ssl"
    output_dir.mkdir(parents=True, exist_ok=True)
    last_checkpoint = output_dir / "last.pt"
    best_checkpoint = output_dir / "best.pt"

    dataset = ContrastiveGCDDataset(
        data_cfg["gcd_root"],
        data_cfg["image_size"],
        data_cfg["classification_classes"],
        data_cfg["val_fraction"],
        cfg["project"]["seed"],
    )
    loader = DataLoader(
        dataset,
        batch_size=ssl_cfg.get("batch_size", 64),
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
        drop_last=True,
    )
    query = ContrastiveCloudEncoder(
        backbone=cls_cfg["backbone"],
        projection_dim=ssl_cfg.get("projection_dim", 128),
        pretrained=True,
    ).to(device)
    key = ContrastiveCloudEncoder(
        backbone=cls_cfg["backbone"],
        projection_dim=ssl_cfg.get("projection_dim", 128),
        pretrained=True,
    ).to(device)
    key.load_state_dict(query.state_dict())
    for param in key.parameters():
        param.requires_grad = False

    optimizer = torch.optim.SGD(
        query.parameters(),
        lr=ssl_cfg.get("lr", 0.03),
        momentum=0.9,
        weight_decay=ssl_cfg.get("weight_decay", 1e-4),
    )
    temperature = ssl_cfg.get("temperature", 0.5)
    momentum = ssl_cfg.get("momentum", 0.999)
    queue_size = ssl_cfg.get("queue_size", 4096)
    projection_dim = ssl_cfg.get("projection_dim", 128)
    queue = F.normalize(torch.randn(queue_size, projection_dim, device=device), dim=1)
    ptr = 0
    scaler = torch.cuda.amp.GradScaler(enabled=device == "cuda")

    best_loss = float("inf")
    start_epoch = 0
    resume_checkpoint = last_checkpoint if last_checkpoint.exists() else best_checkpoint if best_checkpoint.exists() else None
    if resume_checkpoint is not None and not ssl_cfg.get("force_retrain", False):
        checkpoint = load_checkpoint(resume_checkpoint, map_location=device)
        if "model" in checkpoint:
            query.load_state_dict(checkpoint["model"])
        else:
            query.encoder.load_state_dict(checkpoint["encoder"], strict=False)
            query.projector.load_state_dict(checkpoint["projector"], strict=False)
        key.load_state_dict(query.state_dict())
        if "optimizer" in checkpoint:
            try:
                optimizer.load_state_dict(checkpoint["optimizer"])
            except ValueError:
                print("Skipping incompatible CSSL optimizer state; continuing with current optimizer.")
        if "queue" in checkpoint:
            queue = checkpoint["queue"].to(device)
        ptr = int(checkpoint.get("queue_ptr", 0))
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        best_loss = float(checkpoint.get("best_loss", checkpoint.get("loss", best_loss)))
        print(f"Resuming CSSL pretraining from {resume_checkpoint} at epoch {start_epoch + 1}")

    target_epochs = ssl_cfg.get("epochs", 200)
    if start_epoch >= target_epochs:
        print(f"CSSL pretraining already reached target epochs ({target_epochs}).")
        return best_checkpoint if best_checkpoint.exists() else last_checkpoint

    for epoch in range(start_epoch, target_epochs):
        query.train()
        total_loss = 0.0
        for q_images, k_images in tqdm(loader, desc=f"cssl-pretrain-{epoch + 1}"):
            q_images = q_images.to(device, non_blocking=True)
            k_images = k_images.to(device, non_blocking=True)
            with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=device == "cuda"):
                q = F.normalize(query(q_images), dim=1)
                with torch.no_grad():
                    for q_param, k_param in zip(query.parameters(), key.parameters(), strict=True):
                        k_param.data = k_param.data * momentum + q_param.data * (1.0 - momentum)
                    k = F.normalize(key(k_images), dim=1)
                positive = torch.einsum("nc,nc->n", q, k).unsqueeze(1)
                negative = torch.einsum("nc,kc->nk", q, queue.clone().detach())
                logits = torch.cat([positive, negative], dim=1) / temperature
                labels = torch.zeros(logits.shape[0], dtype=torch.long, device=device)
                loss = F.cross_entropy(logits, labels)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            ptr = _dequeue_and_enqueue(queue, k.detach(), ptr)
            total_loss += float(loss.detach().cpu())
        avg_loss = total_loss / max(1, len(loader))
        print(f"cssl_epoch={epoch + 1} loss={avg_loss:.4f}")
        payload = {
            "epoch": epoch,
            "backbone": cls_cfg["backbone"],
            "projection_dim": projection_dim,
            "model": query.state_dict(),
            "encoder": query.encoder.state_dict(),
            "projector": query.projector.state_dict(),
            "optimizer": optimizer.state_dict(),
            "queue": queue.detach().cpu(),
            "queue_ptr": ptr,
            "loss": avg_loss,
            "best_loss": min(best_loss, avg_loss),
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if avg_loss < best_loss:
            best_loss = avg_loss
            save_checkpoint(payload, best_checkpoint)
    return best_checkpoint


def train_classifier(cfg: dict) -> None:
    seed_everything(cfg["project"]["seed"])
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    train_tfms = classification_train_transforms(data_cfg["image_size"], **cfg["augmentation"])
    eval_tfms = eval_transforms(data_cfg["image_size"])
    train_ds = GCDDataset(
        data_cfg["gcd_root"],
        "train",
        train_tfms,
        classes=data_cfg["classification_classes"],
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    val_ds = GCDDataset(
        data_cfg["gcd_root"],
        "val",
        eval_tfms,
        classes=train_ds.classes,
        val_fraction=data_cfg["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    train_loader = DataLoader(
        train_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=True,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    val_loader = DataLoader(
        val_ds,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(train_ds.classes),
        backbone=cls_cfg["backbone"],
        dropout=cls_cfg["dropout"],
    ).to(device)
    ssl_checkpoint = Path(cls_cfg.get("ssl", {}).get("checkpoint", ""))
    if cls_cfg.get("ssl", {}).get("enabled", False) and ssl_checkpoint.exists():
        ssl_payload = load_checkpoint(ssl_checkpoint, map_location=device)
        model.encoder.load_state_dict(ssl_payload["encoder"], strict=False)
        print(f"Loaded CSSL-pretrained encoder from {ssl_checkpoint}")
    criterion = nn.CrossEntropyLoss()
    optimizer_name = str(cls_cfg.get("optimizer", "adamw")).lower()
    if optimizer_name == "sgd":
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=cls_cfg["lr"],
            momentum=cls_cfg.get("momentum", 0.9),
            weight_decay=cls_cfg["weight_decay"],
        )
    elif optimizer_name == "adamw":
        optimizer = torch.optim.AdamW(
            model.parameters(),
            lr=cls_cfg["lr"],
            weight_decay=cls_cfg["weight_decay"],
        )
    else:
        raise ValueError(f"Unsupported classifier optimizer: {optimizer_name}")
    best_f1 = -1.0
    output_dir = Path(cfg["project"]["output_dir"]) / "classifier"
    last_checkpoint = output_dir / "last.pt"
    best_checkpoint = output_dir / "best.pt"
    start_epoch = 0
    resume_checkpoint = last_checkpoint if last_checkpoint.exists() else best_checkpoint if best_checkpoint.exists() else None
    if resume_checkpoint is not None:
        checkpoint = load_checkpoint(resume_checkpoint, map_location=device)
        model.load_state_dict(checkpoint["model"])
        if "optimizer" in checkpoint:
            try:
                optimizer.load_state_dict(checkpoint["optimizer"])
            except ValueError:
                print("Skipping incompatible classifier optimizer state; continuing with current optimizer.")
        start_epoch = int(checkpoint.get("epoch", -1)) + 1
        best_f1 = float(checkpoint.get("best_f1", checkpoint.get("val_metrics", {}).get("f1_macro", -1.0)))
        print(f"Resuming classifier from {resume_checkpoint} at epoch {start_epoch + 1}")

    for epoch in range(start_epoch, cls_cfg["epochs"]):
        model.freeze_encoder(epoch < cls_cfg.get("freeze_backbone_epochs", 0))
        train_metrics = _run_classifier_epoch(
            model, train_loader, criterion, device, optimizer=optimizer, amp=cls_cfg["amp"]
        )
        val_metrics = _run_classifier_epoch(model, val_loader, criterion, device, amp=cls_cfg["amp"])
        print(
            f"epoch={epoch + 1} train_loss={train_metrics['loss']:.4f} "
            f"val_top1={val_metrics['top1']:.4f} val_f1={val_metrics['f1_macro']:.4f}"
        )
        payload = {
            "epoch": epoch,
            "classes": train_ds.classes,
            "backbone": cls_cfg["backbone"],
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "val_metrics": val_metrics,
            "best_f1": max(best_f1, val_metrics["f1_macro"]),
        }
        save_checkpoint(payload, output_dir / "last.pt")
        if val_metrics["f1_macro"] > best_f1:
            best_f1 = val_metrics["f1_macro"]
            save_checkpoint(payload, output_dir / "best.pt")


def evaluate_classifier(cfg: dict) -> dict[str, float]:
    device = get_device(cfg)
    data_cfg = cfg["data"]
    cls_cfg = cfg["classifier"]
    checkpoint = load_checkpoint(cls_cfg["checkpoint"], map_location=device)
    dataset = GCDDataset(
        data_cfg["gcd_root"],
        "test",
        eval_transforms(data_cfg["image_size"]),
        classes=checkpoint["classes"],
    )
    loader = DataLoader(
        dataset,
        batch_size=cls_cfg["batch_size"],
        shuffle=False,
        num_workers=data_cfg["num_workers"],
        pin_memory=device == "cuda",
    )
    model = CloudClassifier(
        num_classes=len(checkpoint["classes"]),
        backbone=checkpoint["backbone"],
        dropout=0.0,
        pretrained=False,
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    metrics = _run_classifier_epoch(model, loader, nn.CrossEntropyLoss(), device, amp=cls_cfg["amp"])
    print(metrics)
    return metrics


In [ ]:
%%writefile cloud_chaser/utils/__init__.py
"""Shared utilities."""


In [ ]:
%%writefile cloud_chaser/utils/checkpoint.py
from __future__ import annotations

from pathlib import Path
from typing import Any

import torch


def save_checkpoint(payload: dict[str, Any], path: str | Path) -> None:
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save(payload, path)


def load_checkpoint(path: str | Path, map_location: str | torch.device = "cpu") -> dict[str, Any]:
    return torch.load(Path(path), map_location=map_location)


In [ ]:
%%writefile cloud_chaser/utils/metrics.py
from __future__ import annotations

import numpy as np
import torch
from sklearn.metrics import accuracy_score, f1_score


def classification_metrics(logits: torch.Tensor, targets: torch.Tensor) -> dict[str, float]:
    preds = logits.argmax(dim=1).detach().cpu().numpy()
    labels = targets.detach().cpu().numpy()
    return {
        "top1": float(accuracy_score(labels, preds)),
        "f1_macro": float(f1_score(labels, preds, average="macro", zero_division=0)),
    }


def binary_iou(pred: np.ndarray, target: np.ndarray) -> float:
    pred = pred.astype(bool)
    target = target.astype(bool)
    union = np.logical_or(pred, target).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(pred, target).sum() / union)


In [ ]:
%%writefile cloud_chaser/utils/seed.py
from __future__ import annotations

import os
import random

import numpy as np
import torch


def seed_everything(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.backends.cudnn.benchmark = True


In [ ]:
%%writefile cloud_chaser/utils/visualization.py
from __future__ import annotations

import cv2
import numpy as np


PALETTE = [
    (42, 157, 143),
    (233, 196, 106),
    (230, 111, 81),
    (38, 70, 83),
    (131, 197, 190),
    (239, 71, 111),
    (17, 138, 178),
]


def overlay_instance(
    image: np.ndarray,
    mask: np.ndarray,
    box: tuple[int, int, int, int],
    label: str,
    score: float,
    color_index: int,
    alpha: float = 0.42,
) -> np.ndarray:
    color = PALETTE[color_index % len(PALETTE)]
    out = image.copy()
    color_layer = np.zeros_like(out)
    color_layer[:, :] = color
    out[mask] = cv2.addWeighted(out, 1 - alpha, color_layer, alpha, 0)[mask]
    x1, y1, x2, y2 = box
    cv2.rectangle(out, (x1, y1), (x2, y2), color, 2)
    text = f"{label} - {score:.0%}"
    (tw, th), _ = cv2.getTextSize(text, cv2.FONT_HERSHEY_SIMPLEX, 0.6, 2)
    y_text = max(0, y1 - th - 8)
    cv2.rectangle(out, (x1, y_text), (x1 + tw + 8, y_text + th + 8), color, -1)
    cv2.putText(
        out,
        text,
        (x1 + 4, y_text + th + 4),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.6,
        (255, 255, 255),
        2,
        cv2.LINE_AA,
    )
    return out


In [ ]:
%%writefile scripts/__init__.py
"""Command-line helper scripts."""


In [ ]:
%%writefile scripts/build_cloud_crops.py
from __future__ import annotations

import argparse
from pathlib import Path

import cv2
import numpy as np
import torch
from tqdm import tqdm

from cloud_chaser.data.augmentations import IMAGENET_MEAN, IMAGENET_STD
from cloud_chaser.data.gcd import IMAGE_EXTENSIONS
from cloud_chaser.models.unet import build_unet
from cloud_chaser.utils.checkpoint import load_checkpoint


def _load_unet(weights: str | Path, device: str) -> torch.nn.Module:
    checkpoint = load_checkpoint(weights, map_location=device)
    model = build_unet(
        architecture=checkpoint.get("architecture", "compact"),
        features=checkpoint.get("features"),
    ).to(device)
    model.load_state_dict(checkpoint["model"])
    model.eval()
    return model


def _unet_boxes(
    model: torch.nn.Module,
    image_rgb: np.ndarray,
    device: str,
    image_size: int,
    threshold: float,
    min_area: int,
) -> list[tuple[int, int, int, int]]:
    h, w = image_rgb.shape[:2]
    resized = cv2.resize(image_rgb, (image_size, image_size), interpolation=cv2.INTER_AREA)
    x = resized.astype(np.float32) / 255.0
    x = (x - np.array(IMAGENET_MEAN, dtype=np.float32)) / np.array(IMAGENET_STD, dtype=np.float32)
    tensor = torch.from_numpy(x.transpose(2, 0, 1))[None].float().to(device)
    with torch.no_grad():
        prob = torch.sigmoid(model(tensor))[0, 0].float().cpu().numpy()
    prob = cv2.resize(prob, (w, h), interpolation=cv2.INTER_LINEAR)
    binary = prob >= threshold
    num_labels, labels = cv2.connectedComponents(binary.astype(np.uint8), connectivity=8)
    boxes: list[tuple[int, int, int, int]] = []
    for component_id in range(1, num_labels):
        mask = labels == component_id
        if int(mask.sum()) < min_area:
            continue
        ys, xs = np.where(mask)
        boxes.append((int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1))
    return boxes


def crop_dataset(
    input_root: Path,
    output_root: Path,
    unet_weights: str,
    threshold: float,
    min_area: int,
    padding: int,
    image_size: int,
    device: str,
) -> None:
    model = _load_unet(unet_weights, device)
    paths = sorted(p for p in input_root.rglob("*") if p.suffix.lower() in IMAGE_EXTENSIONS)
    for path in tqdm(paths, desc="Cropping cloud regions"):
        image_bgr = cv2.imread(str(path))
        if image_bgr is None:
            continue
        image_rgb = cv2.cvtColor(image_bgr, cv2.COLOR_BGR2RGB)
        boxes = _unet_boxes(model, image_rgb, device, image_size, threshold, min_area)
        if not boxes:
            rel = path.relative_to(input_root)
            target = output_root / rel
            target.parent.mkdir(parents=True, exist_ok=True)
            cv2.imwrite(str(target), image_bgr)
            continue
        h, w = image_bgr.shape[:2]
        for idx, box in enumerate(boxes):
            x1, y1, x2, y2 = box
            x1 = max(0, x1 - padding)
            y1 = max(0, y1 - padding)
            x2 = min(w, x2 + padding)
            y2 = min(h, y2 + padding)
            crop = image_bgr[y1:y2, x1:x2]
            rel = path.relative_to(input_root)
            target = output_root / rel.parent / f"{rel.stem}_cloud{idx}{rel.suffix}"
            target.parent.mkdir(parents=True, exist_ok=True)
            if crop.size > 0:
                cv2.imwrite(str(target), crop)


def main() -> None:
    parser = argparse.ArgumentParser(description="Build classifier crop folders from U-Net cloud components.")
    parser.add_argument("--input-root", required=True, type=Path)
    parser.add_argument("--output-root", required=True, type=Path)
    parser.add_argument("--unet-weights", required=True)
    parser.add_argument("--threshold", type=float, default=0.45)
    parser.add_argument("--min-area", type=int, default=256)
    parser.add_argument("--padding", type=int, default=12)
    parser.add_argument("--image-size", type=int, default=224)
    parser.add_argument("--device", default="cuda" if torch.cuda.is_available() else "cpu")
    args = parser.parse_args()
    crop_dataset(
        args.input_root,
        args.output_root,
        args.unet_weights,
        args.threshold,
        args.min_area,
        args.padding,
        args.image_size,
        args.device,
    )


if __name__ == "__main__":
    main()


## Retired ADE20K conversion script

This notebook is now U-Net-only. This former generated-file cell is intentionally retired.


## Retired SWIMSEG polygon conversion script

This notebook now consumes SWIMSEG masks directly through the PyTorch dataset manifest.


In [ ]:
%%writefile scripts/gcd_visual_report.py
from __future__ import annotations

import argparse
import json
import math
import random
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from cloud_chaser.config import get_device, load_config
from cloud_chaser.data.gcd import build_gcd_records
from cloud_chaser.inference_pipeline import CloudIdentifier, display_class_name
from cloud_chaser.utils.seed import seed_everything


def _is_clear_sky(class_name: str) -> bool:
    normalized = class_name.lower().replace("_", "").replace("-", "")
    return "clearsky" in normalized or normalized.endswith("clear") or "clear" in normalized


def _best_image_prediction(predictions) -> tuple[str | None, float]:
    if not predictions:
        return None, 0.0
    best = max(predictions, key=lambda p: p.segmentation_confidence * p.class_confidence)
    return best.class_name, best.segmentation_confidence * best.class_confidence


def _report_path(output_dir: Path, prefix: str, suffix: str) -> Path:
    return output_dir / f"{prefix}_{suffix}"


def _evaluate_cascade(
    cfg: dict,
    output_dir: Path,
    samples: int,
    prefix: str,
) -> tuple[dict, list[dict]]:
    records, classes = build_gcd_records(
        cfg["data"]["gcd_root"],
        split="val",
        classes=cfg["data"].get("classification_classes"),
        val_fraction=cfg["data"]["val_fraction"],
        seed=cfg["project"]["seed"],
    )
    clear_sky = [_is_clear_sky(name) for name in classes]
    identifier = CloudIdentifier(
        unet_weights=cfg["inference"]["unet_weights"],
        classifier_weights=cfg["inference"]["classifier_weights"],
        class_names=classes,
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=cfg["data"]["image_size"],
        half=cfg.get("unet", {}).get("half", True),
        crop_padding=cfg["inference"]["crop_padding"],
    )

    n = len(classes)
    total_by_class = np.zeros(n, dtype=np.int64)
    segmentation_gate_correct_by_class = np.zeros(n, dtype=np.int64)
    detected_by_class = np.zeros(n, dtype=np.int64)
    classified_correct_by_class = np.zeros(n, dtype=np.int64)
    classified_total_by_class = np.zeros(n, dtype=np.int64)
    cascade_correct_by_class = np.zeros(n, dtype=np.int64)
    confusion = np.zeros((n, n), dtype=np.int64)

    details: list[dict] = []
    display_to_idx = {display_class_name(name): idx for idx, name in enumerate(classes)}

    for record in tqdm(records, desc="GCD cascade validation"):
        true_idx = int(record.label)
        true_name = display_class_name(classes[true_idx])
        expects_cloud = not clear_sky[true_idx]
        _, predictions = identifier.predict(record.path)
        has_detection = len(predictions) > 0
        pred_name, pred_score = _best_image_prediction(predictions)
        pred_idx = display_to_idx.get(pred_name) if pred_name is not None else None

        segmentation_gate_correct = has_detection if expects_cloud else not has_detection
        classification_correct = pred_idx == true_idx if has_detection and pred_idx is not None else False
        if expects_cloud:
            cascade_correct = has_detection and classification_correct
        else:
            cascade_correct = not has_detection

        total_by_class[true_idx] += 1
        detected_by_class[true_idx] += int(has_detection)
        segmentation_gate_correct_by_class[true_idx] += int(segmentation_gate_correct)
        cascade_correct_by_class[true_idx] += int(cascade_correct)
        if has_detection and pred_idx is not None:
            classified_total_by_class[true_idx] += 1
            classified_correct_by_class[true_idx] += int(classification_correct)
            confusion[true_idx, pred_idx] += 1

        details.append(
            {
                "path": str(record.path),
                "true_class": true_name,
                "expects_cloud": expects_cloud,
                "has_detection": has_detection,
                "segmentation_gate_correct": bool(segmentation_gate_correct),
                "predicted_class": pred_name,
                "prediction_score": float(pred_score),
                "classification_correct": bool(classification_correct),
                "cascade_correct": bool(cascade_correct),
                "num_detections": len(predictions),
            }
        )

    segmentation_gate_accuracy = float(segmentation_gate_correct_by_class.sum() / max(total_by_class.sum(), 1))
    conditional_classification_accuracy = float(
        classified_correct_by_class.sum() / max(classified_total_by_class.sum(), 1)
    )
    cascade_accuracy = float(cascade_correct_by_class.sum() / max(total_by_class.sum(), 1))

    metrics = {
        "split": "val",
        "segmenter": "unet",
        "num_images": int(total_by_class.sum()),
        "classes": classes,
        "note": (
            "GCD has image-level class labels but no cloud masks. U-Net segmentation is evaluated as an "
            "image-level cascade gate: non-clearsky classes should produce at least one cloud "
            "detection, while clearsky should produce none."
        ),
        "segmentation_gate_accuracy": segmentation_gate_accuracy,
        "classifier_accuracy_given_detection": conditional_classification_accuracy,
        "cascade_accuracy": cascade_accuracy,
        "total_by_class": total_by_class.tolist(),
        "detected_by_class": detected_by_class.tolist(),
        "segmentation_gate_correct_by_class": segmentation_gate_correct_by_class.tolist(),
        "classified_total_by_class": classified_total_by_class.tolist(),
        "classified_correct_by_class": classified_correct_by_class.tolist(),
        "cascade_correct_by_class": cascade_correct_by_class.tolist(),
        "confusion_matrix_given_detection": confusion.tolist(),
        "details": details,
    }
    _report_path(output_dir, prefix, "metrics.json").write_text(json.dumps(metrics, indent=2))
    _plot_cascade_bars(metrics, output_dir, prefix)
    overlay_path = _make_overlay_grid(cfg, output_dir, details, samples, prefix)
    return metrics, details if overlay_path else details


def _plot_cascade_bars(metrics: dict, output_dir: Path, prefix: str) -> None:
    labels = [display_class_name(name) for name in metrics["classes"]]
    total = np.array(metrics["total_by_class"])
    segmentation_gate_correct = np.array(metrics["segmentation_gate_correct_by_class"])
    classified_total = np.array(metrics["classified_total_by_class"])
    classified_correct = np.array(metrics["classified_correct_by_class"])
    cascade_correct = np.array(metrics["cascade_correct_by_class"])

    x = np.arange(len(labels))
    width = 0.26
    fig, ax = plt.subplots(figsize=(13, 6.5))
    b1 = ax.bar(x - width, segmentation_gate_correct, width, color="#457b9d", label="U-Net gate correct")
    b2 = ax.bar(x, classified_correct, width, color="#2a9d8f", label="Classified correct after detection")
    b3 = ax.bar(x + width, cascade_correct, width, color="#e76f51", label="End-to-end correct")
    ax.set_title(
        "GCD validation cascade: U-Net cloud segmentation -> cloud type classification\n"
        f"seg-gate={metrics['segmentation_gate_accuracy']:.1%}, "
        f"cls|det={metrics['classifier_accuracy_given_detection']:.1%}, "
        f"end-to-end={metrics['cascade_accuracy']:.1%}"
    )
    ax.set_ylabel("Images")
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)

    for bars, denominators in [(b1, total), (b2, classified_total), (b3, total)]:
        for bar, denom in zip(bars, denominators, strict=False):
            value = int(bar.get_height())
            ax.text(
                bar.get_x() + bar.get_width() / 2,
                value + max(total.max() * 0.015, 1),
                f"{value}/{int(denom)}",
                ha="center",
                va="bottom",
                fontsize=8,
                rotation=90,
            )
    fig.tight_layout()
    fig.savefig(_report_path(output_dir, prefix, "bar.png"), dpi=180)
    plt.close(fig)


def _select_samples(details: list[dict], samples: int, seed: int) -> list[dict]:
    rng = random.Random(seed)
    groups = [
        [d for d in details if d["has_detection"] and d["cascade_correct"]],
        [d for d in details if d["has_detection"] and not d["cascade_correct"]],
        [d for d in details if not d["has_detection"] and d["expects_cloud"]],
        [d for d in details if not d["has_detection"] and not d["expects_cloud"]],
    ]
    selected: list[dict] = []
    for group in groups:
        rng.shuffle(group)
        selected.extend(group[: max(1, samples // len(groups))])
    if len(selected) < samples:
        remaining = [d for d in details if d not in selected]
        rng.shuffle(remaining)
        selected.extend(remaining[: samples - len(selected)])
    return selected[:samples]


def _make_overlay_grid(
    cfg: dict,
    output_dir: Path,
    details: list[dict],
    samples: int,
    prefix: str,
) -> Path | None:
    selected = _select_samples(details, samples, cfg["project"]["seed"])
    if not selected:
        return None

    classes = cfg["data"].get("classification_classes")
    identifier = CloudIdentifier(
        unet_weights=cfg["inference"]["unet_weights"],
        classifier_weights=cfg["inference"]["classifier_weights"],
        class_names=classes,
        unet_threshold=cfg.get("unet", {}).get("threshold", 0.45),
        unet_min_area=cfg.get("unet", {}).get("min_area", 256),
        device=get_device(cfg),
        image_size=cfg["data"]["image_size"],
        half=cfg.get("unet", {}).get("half", True),
        crop_padding=cfg["inference"]["crop_padding"],
    )

    overlays: list[np.ndarray] = []
    titles: list[str] = []
    for item in tqdm(selected, desc="GCD cascade overlay samples"):
        overlay_bgr, predictions = identifier.predict(item["path"])
        overlay_rgb = cv2.cvtColor(overlay_bgr, cv2.COLOR_BGR2RGB)
        if predictions:
            pred = ", ".join(f"{p.class_name} {p.class_confidence:.0%}" for p in predictions[:2])
        else:
            pred = "No cloud detection"
        status = "OK" if item["cascade_correct"] else "FAIL"
        overlays.append(overlay_rgb)
        titles.append(f"{status} | True: {item['true_class']}\nPred: {pred}")

    cols = min(3, len(overlays))
    rows = math.ceil(len(overlays) / cols)
    fig, axes = plt.subplots(rows, cols, figsize=(5.0 * cols, 4.6 * rows))
    axes_array = np.atleast_1d(axes).reshape(rows, cols)
    for ax in axes_array.ravel():
        ax.axis("off")
    for ax, image, title in zip(axes_array.ravel(), overlays, titles, strict=False):
        ax.imshow(image)
        ax.set_title(title, fontsize=10)
    fig.tight_layout()
    output_path = _report_path(output_dir, prefix, "overlay_samples.jpg")
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Evaluate the GCD U-Net segmentation/classification cascade.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--output-dir", default="reports")
    parser.add_argument("--samples", type=int, default=9)
    parser.add_argument("--prefix", default=None)
    args = parser.parse_args()

    cfg = load_config(args.config)
    seed_everything(cfg["project"]["seed"])
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    prefix = args.prefix or "gcd_val_unet_cascade"

    metrics, _ = _evaluate_cascade(cfg, output_dir, args.samples, prefix)
    print("segmenter=unet")
    print(f"saved={_report_path(output_dir, prefix, 'bar.png')}")
    print(f"saved={_report_path(output_dir, prefix, 'overlay_samples.jpg')}")
    print(f"saved={_report_path(output_dir, prefix, 'metrics.json')}")
    print(f"GCD U-Net segmentation gate accuracy={metrics['segmentation_gate_accuracy']:.4f}")
    print(f"GCD classifier accuracy given detection={metrics['classifier_accuracy_given_detection']:.4f}")
    print(f"GCD cascade accuracy={metrics['cascade_accuracy']:.4f}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile scripts/unet_ablation_report.py
from __future__ import annotations

import argparse
import copy
import csv
import json
from pathlib import Path

import matplotlib.pyplot as plt

from cloud_chaser.config import load_config
from cloud_chaser.training import evaluate_unet_segmenter, train_unet_segmenter
from scripts.gcd_visual_report import _evaluate_cascade


def _variant_cfg(base_cfg: dict, variant: dict) -> dict:
    cfg = copy.deepcopy(base_cfg)
    cfg["unet"]["architecture"] = variant["architecture"]
    cfg["unet"]["features"] = variant["features"]
    checkpoint = Path(cfg["project"]["output_dir"]) / "unet" / variant["name"] / "best.pt"
    cfg["unet"]["checkpoint"] = str(checkpoint)
    cfg["inference"]["unet_weights"] = str(checkpoint)
    return cfg


def _plot_summary(rows: list[dict], output_dir: Path) -> Path:
    labels = [row["name"] for row in rows]
    miou = [float(row["swimseg_miou"]) for row in rows]
    dice = [float(row["swimseg_dice"]) for row in rows]
    cascade = [float(row["cascade_accuracy"]) for row in rows]

    fig, ax = plt.subplots(figsize=(13, 5.8))
    x = list(range(len(rows)))
    width = 0.25
    ax.bar([i - width for i in x], miou, width, label="SWIMSEG mIoU")
    ax.bar(x, dice, width, label="SWIMSEG Dice")
    ax.bar([i + width for i in x], cascade, width, label="GCD cascade")
    ax.set_ylim(0, 1)
    ax.set_xticks(x)
    ax.set_xticklabels(labels, rotation=25, ha="right")
    ax.set_ylabel("Score")
    ax.set_title("Six U-Net pipeline comparison")
    ax.grid(axis="y", alpha=0.25)
    ax.legend(frameon=False)
    for container in ax.containers:
        ax.bar_label(container, labels=[f"{v:.2f}" for v in container.datavalues], fontsize=7, padding=2)
    fig.tight_layout()
    output_path = output_dir / "unet_ablation_summary.png"
    fig.savefig(output_path, dpi=180)
    plt.close(fig)
    return output_path


def main() -> None:
    parser = argparse.ArgumentParser(description="Train/evaluate the six U-Net research pipelines.")
    parser.add_argument("--config", default="configs/default.yaml")
    parser.add_argument("--output-dir", default="../results/reports/unet_ablation")
    parser.add_argument("--skip-train", action="store_true")
    parser.add_argument("--samples", type=int, default=9)
    args = parser.parse_args()

    base_cfg = load_config(args.config)
    output_dir = Path(args.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    rows: list[dict] = []

    for variant in base_cfg["experiments"]["unet_variants"]:
        name = variant["name"]
        cfg = _variant_cfg(base_cfg, variant)
        if not args.skip_train:
            train_unet_segmenter(cfg, experiment_name=name, unet_override=variant)
        unet_metrics = evaluate_unet_segmenter(cfg, checkpoint_path=cfg["unet"]["checkpoint"], unet_override=variant)
        prefix = f"{name}_gcd"
        cascade_metrics, _ = _evaluate_cascade(cfg, output_dir, args.samples, prefix)
        row = {
            "name": name,
            "architecture": variant["architecture"],
            "features": " ".join(str(v) for v in variant["features"]),
            "checkpoint": cfg["unet"]["checkpoint"],
            "swimseg_loss": unet_metrics["loss"],
            "swimseg_miou": unet_metrics["miou"],
            "swimseg_dice": unet_metrics["dice"],
            "segmentation_gate_accuracy": cascade_metrics["segmentation_gate_accuracy"],
            "classifier_accuracy_given_detection": cascade_metrics["classifier_accuracy_given_detection"],
            "cascade_accuracy": cascade_metrics["cascade_accuracy"],
        }
        rows.append(row)
        (output_dir / f"{name}_summary.json").write_text(json.dumps(row, indent=2), encoding="utf-8")

    summary_path = output_dir / "unet_ablation_summary.csv"
    with summary_path.open("w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0].keys()))
        writer.writeheader()
        writer.writerows(rows)
    plot_path = _plot_summary(rows, output_dir)
    print(f"saved={summary_path}")
    print(f"saved={plot_path}")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile configs/default.yaml
project:
  name: cloud-chaser
  seed: 42
  device: cuda
  output_dir: ../results

data:
  swimseg_root: ../data/swimseg-2
  gcd_root: ../data/GCD
  prepared_seg_dir: ../data/processed/swimseg_cloud_masks
  num_workers: 8
  image_size: 224
  val_fraction: 0.15
  seg_val_fraction: 0.10
  seg_test_fraction: 0.10
  classification_classes:
    - 1_cumulus
    - 2_altocumulus
    - 3_cirrus
    - 4_clearsky
    - 5_stratocumulus
    - 6_cumulonimbus
    - 7_mixed
  min_mask_area: 96
  swimseg_invert_masks: false

augmentation:
  random_shadow_p: 0.25
  gaussian_blur_p: 0.20
  hflip_p: 0.50
  vflip_p: 0.15

unet:
  checkpoint: ../results/unet/best.pt
  architecture: compact
  features: [32, 64, 128, 256]
  epochs: 60
  batch_size: 8
  lr: 0.0003
  weight_decay: 0.00001
  threshold: 0.45
  min_area: 256
  half: true

experiments:
  unet_variants:
    - name: baseline_a_compact
      architecture: compact
      features: [32, 64, 128, 256]
    - name: baseline_b_medium
      architecture: standard
      features: [64, 128, 256, 512]
    - name: model_c_dilated
      architecture: dilated
      features: [64, 128, 256, 512]
    - name: model_d_dilated_aspp
      architecture: dilated_aspp
      features: [64, 128, 256, 512]
    - name: model_e_bicubic
      architecture: bicubic
      features: [64, 128, 256, 512]
    - name: model_f_improved
      architecture: improved
      features: [64, 128, 256, 512]

classifier:
  backbone: resnet50
  batch_size: 16
  epochs: 200
  lr: 0.01
  weight_decay: 0.0001
  optimizer: sgd
  momentum: 0.9
  dropout: 0.2
  amp: true
  freeze_backbone_epochs: 2
  checkpoint: ../results/classifier/best.pt
  ssl:
    enabled: true
    checkpoint: ../results/classifier_ssl/best.pt
    projection_dim: 128
    queue_size: 4096
    temperature: 0.5
    momentum: 0.999
    batch_size: 64
    epochs: 200
    lr: 0.03
    weight_decay: 0.0001

inference:
  unet_weights: ../results/unet/best.pt
  classifier_weights: ../results/classifier/best.pt
  output_dir: ../results/inference
  crop_padding: 12
  mask_alpha: 0.42


## Install Dependencies

In [ ]:
import torch, sys
print('Python:', sys.version)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none')


In [ ]:
import subprocess
subprocess.run(['python', '-m', 'pip', 'install', '-q', '--upgrade', 'pip'], check=True)
subprocess.run(['python', '-m', 'pip', 'install', '-q', '-e', '.'], check=True)


## Configure Dataset Paths

Attach `antigs/skyimage-dataset` and your GCD Kaggle dataset, then run this cell. It auto-discovers common folder names under `/kaggle/input`.

In [ ]:
from pathlib import Path
import yaml
import torch

cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
KAGGLE_INPUT = Path('/kaggle/input')
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def image_count(path):
    return sum(1 for p in Path(path).rglob('*') if p.suffix.lower() in IMAGE_EXTS)

def find_swimseg_root():
    explicit = [
        Path('/kaggle/input/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/swimseg-2'),
        Path('/kaggle/input/datasets/antigs/skyimage-dataset/SWIMSEG-2'),
        Path('/kaggle/input/datasets/skyimage-dataset/swimseg-2'),
    ]
    for path in explicit:
        if path.exists():
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir():
                continue
            if 'swimseg' not in str(path).lower():
                continue
            count = image_count(path)
            if count:
                candidates.append((count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], len(item[1].parts), str(item[1])))[0][1])
    return ''

def has_split_dir(path, split_names):
    return any((path / name).exists() and (path / name).is_dir() for name in split_names)

def is_not_segmentation_dataset(path):
    text = str(path).lower()
    banned = ['swimseg', 'skyimage', 'sky-image', 'segmentation', 'mask', 'latest-output']
    return not any(token in text for token in banned)

def find_gcd_root():
    explicit = [
        Path('/kaggle/input/gcd/GCD'),
        Path('/kaggle/input/gcd'),
        Path('/kaggle/input/global-cloud-dataset/GCD'),
        Path('/kaggle/input/global-cloud-dataset'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset/GCD'),
        Path('/kaggle/input/datasets/alopixalopix/gcd-dataset'),
    ]
    for path in explicit:
        if path.exists() and is_not_segmentation_dataset(path) and has_split_dir(path, ['train', 'Train', 'training', 'Training']):
            return str(path)
    candidates = []
    if KAGGLE_INPUT.exists():
        for path in KAGGLE_INPUT.rglob('*'):
            if not path.is_dir() or not is_not_segmentation_dataset(path):
                continue
            if has_split_dir(path, ['train', 'Train', 'training', 'Training']):
                class_root = next((path / n for n in ['train', 'Train', 'training', 'Training'] if (path / n).exists()), None)
            else:
                class_root = path
            class_dirs = [p for p in class_root.iterdir() if p.is_dir()] if class_root and class_root.exists() else []
            names = ' '.join(p.name.lower().replace('_', '') for p in class_dirs)
            cloud_hits = sum(1 for token in ['cumulus', 'altocumulus', 'cirrus', 'clearsky', 'stratocumulus', 'cumulonimbus', 'mixed'] if token in names)
            count = image_count(class_root) if class_root else 0
            if len(class_dirs) >= 2 and count and cloud_hits >= 2:
                candidates.append((cloud_hits, count, path))
    if candidates:
        return str(sorted(candidates, key=lambda item: (-item[0], -item[1], len(item[2].parts), str(item[2])))[0][2])
    return ''

cfg['data']['swimseg_root'] = find_swimseg_root()
cfg['data']['gcd_root'] = find_gcd_root()
cfg['data']['prepared_seg_dir'] = '/kaggle/working/swimseg_masks'
cfg['project']['output_dir'] = '/kaggle/working/runs'
cfg['project']['device'] = 'cuda' if torch.cuda.is_available() else 'cpu'
cfg['unet']['epochs'] = 60
cfg['classifier']['epochs'] = 80
cfg['inference']['unet_weights'] = '/kaggle/working/runs/unet/best.pt'
cfg['inference']['classifier_weights'] = '/kaggle/working/runs/classifier/best.pt'
cfg['classifier']['checkpoint'] = '/kaggle/working/runs/classifier/best.pt'
cfg['classifier'].setdefault('ssl', {})['checkpoint'] = '/kaggle/working/runs/classifier_ssl/best.pt'
cfg['unet']['checkpoint'] = '/kaggle/working/runs/unet/best.pt'

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())
for label, key in [('SWIMSEG', 'swimseg_root'), ('GCD', 'gcd_root')]:
    value = cfg['data'][key]
    path = Path(value) if value else Path('__missing__')
    print(f'{label}: {value or "NOT FOUND"} exists={path.exists()}')

if not cfg['data']['swimseg_root'] or not Path(cfg['data']['swimseg_root']).exists():
    print('\nAvailable /kaggle/input folders:')
    for path in sorted(KAGGLE_INPUT.glob('*')) if KAGGLE_INPUT.exists() else []:
        print(' -', path)
    raise FileNotFoundError('Attach the Kaggle Sky-Image Dataset, or update data.swimseg_root above.')


In [ ]:
from pathlib import Path
import shutil

LATEST_OUTPUT_ROOT = Path('/kaggle/input/datasets/alopixalopix/latest-output')
RUNS_ROOT = Path('/kaggle/working/runs')
CHECKPOINTS_ROOT = Path('/kaggle/working/checkpoints')

UNET_VARIANTS = [
    'baseline_a_compact',
    'baseline_b_medium',
    'model_c_dilated',
    'model_d_dilated_aspp',
    'model_e_bicubic',
    'model_f_improved',
]

def find_artifact(root, candidates, predicate):
    for candidate in candidates:
        path = root / candidate
        if path.exists():
            return path
    if root.exists():
        matches = [p for p in root.rglob('*') if p.is_file() and predicate(p)]
        if matches:
            return sorted(matches, key=lambda p: (len(p.parts), str(p)))[0]
    return None

def restore_artifact(src, dst):
    if src is None:
        print(f'No saved artifact found for {dst}')
        return False
    if dst.exists():
        print(f'Already present, keeping: {dst}')
        return True
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(src, dst)
    print(f'Restored {dst} from {src}')
    return True

def restore_stage(stage, dst_dir, extra_roots=None):
    extra_roots = extra_roots or []
    for name in ['last.pt', 'best.pt']:
        candidate_paths = [
            Path(f'runs/{stage}/{name}'),
            Path(f'cloud-chaser/runs/{stage}/{name}'),
            Path(f'{stage}/{name}'),
        ]
        candidate_paths.extend(Path(root) / name for root in extra_roots)
        src = find_artifact(
            LATEST_OUTPUT_ROOT,
            candidate_paths,
            lambda p, stage=stage, name=name: p.name == name and stage in str(p).lower(),
        )
        restore_artifact(src, dst_dir / name)

if LATEST_OUTPUT_ROOT.exists():
    print('Restoring saved model artifacts from', LATEST_OUTPUT_ROOT)
    restore_stage('unet', RUNS_ROOT / 'unet')
    for variant_name in UNET_VARIANTS:
        restore_stage(
            variant_name,
            RUNS_ROOT / 'unet' / variant_name,
            extra_roots=[
                Path('runs/unet') / variant_name,
                Path('cloud-chaser/runs/unet') / variant_name,
                Path('unet') / variant_name,
            ],
        )
    restore_stage('classifier_ssl', RUNS_ROOT / 'classifier_ssl')
    restore_stage('classifier', RUNS_ROOT / 'classifier')
    torchscript_src = find_artifact(
        LATEST_OUTPUT_ROOT,
        [Path('classifier.torchscript'), Path('artifacts/classifier.torchscript'), Path('cloud-chaser/classifier.torchscript')],
        lambda p: p.name == 'classifier.torchscript',
    )
    restore_artifact(torchscript_src, Path('/kaggle/working/artifacts/classifier.torchscript'))
else:
    print('No latest-output checkpoint dataset attached at', LATEST_OUTPUT_ROOT)

CHECKPOINTS_ROOT.mkdir(parents=True, exist_ok=True)
print('Checkpoint restore directory:', RUNS_ROOT)


## Dataset Tree Debug

Run this if path discovery fails or if the SWIMSEG/GCD folder names differ.

In [ ]:
from pathlib import Path
for root in sorted(Path('/kaggle/input').glob('*')):
    print(root)
    for child in sorted(root.rglob('*'))[:80]:
        print('  ', child.relative_to(root))


## GCD Dataset Sanity Check


In [ ]:
from pathlib import Path
import yaml
cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
gcd_value = cfg['data'].get('gcd_root')
print('GCD root:', gcd_value or 'NOT FOUND')
if not gcd_value:
    print('No GCD classification dataset was detected. Attach GCD or set data.gcd_root manually.')
else:
    gcd_root = Path(gcd_value)
    print('exists=', gcd_root.exists())
    for split_name in ['train', 'Train', 'training', 'Training', 'test', 'Test', 'val', 'validation']:
        split_dir = gcd_root / split_name
        if split_dir.exists():
            print('\n', split_dir)
            for class_dir in sorted([p for p in split_dir.iterdir() if p.is_dir()]):
                count = sum(1 for p in class_dir.rglob('*') if p.suffix.lower() in {'.jpg', '.jpeg', '.png', '.bmp', '.webp'})
                print(' ', class_dir.name, count)


## Prepare SWIMSEG-2 Masks

U-Net training prepares the SWIMSEG image-mask manifest automatically, so there is no separate conversion step.


In [ ]:
print('SWIMSEG mask preparation is handled inside train.py unet.')


## Train U-Net Segmenter


In [ ]:
from pathlib import Path
import subprocess

assert Path('train.py').exists(), 'train.py was not written; rerun notebook from the top.'
print('Training/extending U-Net segmenter. Existing last.pt will be resumed automatically if present.')
subprocess.run(['python', 'train.py', 'unet', '--config', 'configs/default.yaml'], check=True)


## CSSL Classifier Pretraining

Pretrains the ResNet50 encoder with the contrastive self-supervised method from the attached paper.


In [ ]:
from pathlib import Path
import subprocess
import yaml

cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
gcd_root = cfg['data'].get('gcd_root')
print('Configured GCD root:', gcd_root or 'NOT FOUND')
if not gcd_root or not Path(gcd_root).exists():
    print('Skipping CSSL pretraining: no GCD classification dataset is attached.')
else:
    subprocess.run(['python', 'train.py', 'classifier-ssl', '--config', 'configs/default.yaml'], check=True)


## Fine-Tune GCD Classifier

Fine-tunes the classifier head and encoder using the CSSL-pretrained encoder when available.


In [ ]:
from pathlib import Path
import subprocess
import yaml

cfg = yaml.safe_load(Path('configs/default.yaml').read_text())
gcd_root = cfg['data'].get('gcd_root')
if not gcd_root or not Path(gcd_root).exists():
    print('Skipping classifier training: no GCD classification dataset is attached.')
else:
    print('Training/extending classifier. Existing last.pt will be resumed automatically if present.')
    subprocess.run(['python', 'train.py', 'classifier', '--config', 'configs/default.yaml'], check=True)


## Six U-Net Pipeline Comparison

Trains/evaluates Baseline A through Model F and writes the comparison report. Existing checkpoints resume automatically.


In [ ]:
from pathlib import Path
import subprocess
from IPython.display import Image, display

REPORT_DIR = Path('/kaggle/working/reports/unet_ablation')
subprocess.run([
    'python',
    'scripts/unet_ablation_report.py',
    '--config',
    'configs/default.yaml',
    '--output-dir',
    str(REPORT_DIR),
    '--samples',
    '9',
], check=True)
summary_plot = REPORT_DIR / 'unet_ablation_summary.png'
if summary_plot.exists():
    display(Image(filename=str(summary_plot)))
print('Ablation outputs:', REPORT_DIR)


## Evaluate

## Snapshot Checkpoints


In [ ]:
from pathlib import Path
import shutil

CHECKPOINTS = Path('/kaggle/working/checkpoints')
CHECKPOINTS.mkdir(parents=True, exist_ok=True)
items = {
    'unet_best.pt': Path('/kaggle/working/runs/unet/best.pt'),
    'unet_last.pt': Path('/kaggle/working/runs/unet/last.pt'),
    'classifier_best.pt': Path('/kaggle/working/runs/classifier/best.pt'),
    'classifier_last.pt': Path('/kaggle/working/runs/classifier/last.pt'),
}
for name, src in items.items():
    if src.exists():
        dst = CHECKPOINTS / name
        shutil.copy2(src, dst)
        print('checkpoint=', dst)
    else:
        print('missing checkpoint source=', src)


In [ ]:
from pathlib import Path
import subprocess

best_classifier = Path('/kaggle/working/runs/classifier/best.pt')
best_unet = Path('/kaggle/working/runs/unet/best.pt')

if best_unet.exists():
    print(f'Evaluating U-Net checkpoint: {best_unet}')
    result = subprocess.run(['python', 'train.py', 'eval-unet', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('U-Net evaluation failed, but the U-Net checkpoint exists. Continuing notebook run.')
else:
    print('Skipping U-Net evaluation: missing', best_unet)

if best_classifier.exists():
    print(f'Evaluating classifier checkpoint: {best_classifier}')
    result = subprocess.run(['python', 'train.py', 'eval-classifier', '--config', 'configs/default.yaml'], check=False)
    if result.returncode:
        print('Classifier evaluation failed, but the classifier checkpoint exists. Continuing notebook run.')
else:
    print('Skipping classifier evaluation: missing', best_classifier)


## GCD U-Net Cascade Report

Evaluates the U-Net segmentation gate and the downstream cloud-type classifier on the GCD validation split.


In [ ]:
from pathlib import Path
import json
import subprocess
from IPython.display import Image, display

REPORT_DIR = Path('/kaggle/working/reports')
REPORT_DIR.mkdir(parents=True, exist_ok=True)
subprocess.run([
    'python',
    'scripts/gcd_visual_report.py',
    '--config',
    'configs/default.yaml',
    '--output-dir',
    str(REPORT_DIR),
    '--samples',
    '9',
], check=True)

metrics_path = REPORT_DIR / 'gcd_val_unet_cascade_metrics.json'
if metrics_path.exists():
    metrics = json.loads(metrics_path.read_text())
    print(
        f"seg-gate={metrics['segmentation_gate_accuracy']:.1%} | "
        f"cls|det={metrics['classifier_accuracy_given_detection']:.1%} | "
        f"cascade={metrics['cascade_accuracy']:.1%}"
    )

for image_path in [REPORT_DIR / 'gcd_val_unet_cascade_bar.png', REPORT_DIR / 'gcd_val_unet_cascade_overlay_samples.jpg']:
    if image_path.exists():
        print(image_path)
        display(Image(filename=str(image_path)))
    else:
        print('Missing report image:', image_path)


## Inference Example

Change `IMAGE_PATH` to a real image in `/kaggle/input` or `/kaggle/working`.

In [ ]:
from pathlib import Path
import subprocess
import yaml
from IPython.display import Image, display

IMAGE_PATH = Path('/kaggle/input/datasets/alopixalopix/examples/cirocumulus.jpg')
OUTPUT_PATH = Path('/kaggle/working/prediction.jpg')
UNET_BEST = Path('/kaggle/working/runs/unet/best.pt')
CLASSIFIER_BEST = Path('/kaggle/working/runs/classifier/best.pt')
CLASSIFIER_TS = Path('/kaggle/working/artifacts/classifier.torchscript')

classifier_artifact = CLASSIFIER_TS if CLASSIFIER_TS.exists() else CLASSIFIER_BEST
required = [UNET_BEST, classifier_artifact, IMAGE_PATH]
missing = [str(p) for p in required if not p.exists()]
if missing:
    print('Skipping inference because required files are missing:', missing)
else:
    cfg_path = Path('configs/default.yaml')
    cfg = yaml.safe_load(cfg_path.read_text())
    cfg['inference']['unet_weights'] = str(UNET_BEST)
    cfg['inference']['classifier_weights'] = str(classifier_artifact)
    cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
    print('Using classifier artifact:', classifier_artifact)
    subprocess.run(['python', 'inference.py', '--config', 'configs/default.yaml', '--image', str(IMAGE_PATH), '--output', str(OUTPUT_PATH)], check=True)
    display(Image(filename=str(OUTPUT_PATH)))


## Export Artifacts

In [ ]:
from pathlib import Path
import subprocess

ARTIFACTS = Path('/kaggle/working/artifacts')
ARTIFACTS.mkdir(parents=True, exist_ok=True)
best_unet = Path('/kaggle/working/runs/unet/best.pt')
best_classifier = Path('/kaggle/working/runs/classifier/best.pt')
unet_ts = ARTIFACTS / 'unet.torchscript'
classifier_ts = ARTIFACTS / 'classifier.torchscript'

if best_unet.exists():
    if unet_ts.exists():
        print('Skipping U-Net TorchScript export; found', unet_ts)
    else:
        result = subprocess.run(
            ['python', 'export.py', 'unet', '--config', 'configs/default.yaml', '--format', 'torchscript', '--output', str(unet_ts)],
            check=False,
        )
        if result.returncode:
            print('U-Net TorchScript export failed; continuing because best.pt is available.')
else:
    print('Skipping U-Net export: missing', best_unet)

if best_classifier.exists():
    if classifier_ts.exists():
        print('Skipping classifier TorchScript export; found', classifier_ts)
    else:
        result = subprocess.run(
            ['python', 'export.py', 'classifier', '--config', 'configs/default.yaml', '--format', 'torchscript', '--output', str(classifier_ts)],
            check=False,
        )
        if result.returncode:
            print('Classifier TorchScript export failed; continuing because best.pt is available.')
else:
    print('Skipping classifier export: missing', best_classifier)
